# Especialização Multi-Resolução e Multi-Escala MobileNetV2 ($\alpha \in \{0.35, 0.50, 0.75\}$) com QAT Int8 para Diagnóstico Fitopatológico Embarcado no ESP32-S3 (v10)

**Objetivo:** Evoluir a arquitetura de treinamento da `v09` respeitando as particularidades representacionais de cada recorte da **MobileNetV2** ($\alpha \in \{0.35, 0.50, 0.75\}$) por meio de **escalonamento composto de resolução de entrada ($192\times 192$ e $224\times 224$)**, **regularização espacial 2D (`SpatialDropout2D`)** e **treinamento em 4 estágios com descongelamento progressivo profundo + *Clean Cooldown* pré-QAT**, mantendo compatibilidade binária estrita com o **ESP32-S3 N8R16** (`esp-nn` + `MicroMutableOpResolver<6>`).

---

## 🔍 1. Avaliação Crítica dos Ciclos Anteriores (`v07` $\rightarrow$ `v08` $\rightarrow$ `v09`)

| Versão | Estratégia Principal | Principais Resultados Observados | Gargalos Identificados |
| :--- | :--- | :--- | :--- |
| **`v07`** | CNN Customizada 4-blocos vs. MobileNetV2 ($\alpha=0.35$, $160\times 160$, `bilinear`) em dataset bruto | CNN colapsou no QAT (`85,32%` F32 $\rightarrow$ `44,11%` Int8); MobileNetV2 atingiu `91,03%` F32 / `91,18%` Int8 (`624,3 KB`), mas `smut` travou em `61,9%` (val) e `4%–6%` no ESP32-S3. | *Data leakage* (747 duplicatas) + 306 colisões MD5 cruzadas entre classes + `bilinear` divergente do `BoxAvg`. |
| **`v08`** | Auditoria MD5 (`5.389` imgs puras: `4.313` treino / `1.076` val) + `interpolation="area"` + Focal Loss + BN congelado | Erradicou 100% da confusão `smut` $\leftrightarrow$ `Pokkah Boeng`; **`91,09%`** em 550 limpas (`Modo 7`), **`89,41%`** em `val/` e **`89,82%`** no ESP32-S3 físico (`283,4 ms` @ 240 MHz + PSRAM 80 MHz DDR). | Restrito apenas à escala única $\alpha=0.35$ em $160\times 160$ sem regularização espacial para capacidades maiores. |
| **`v09`** | Ablação multi-escala ($\alpha \in \{0.35, 0.50, 0.75\}$) em $160\times 160$ fixo c/ MixUp, ruído de sensor e congelamento estático | Descontinuou CNN Customizada; gerou `623,9 KB` ($\alpha=0.35$), `965,3 KB` ($\alpha=0.50$) e `1.720,2 KB` ($\alpha=0.75$) com $\Delta_{\text{QAT}} \le 0,3\%$. | **1) Fome de resolução ($5\times 5$ no GAP)** em redes largas; **2) Congelamento estático** sem fase de refinamento limpo antes do QAT; **3) Ausência de `SpatialDropout2D`**. |

---

## 🚀 2. Inovações Arquiteturais da Versão `v10` (Por Recorte $\alpha$)

Em vez de forçar três capacidades distintas ($\alpha=0.35$, $0.50$ e $0.75$) à mesma entrada fixa de $160\times 160$ (que comprime a imagem a um mapa de apenas $5\times 5 = 25$ células antes do `GlobalAveragePooling2D`), a **`v10`** aplica **co-adaptação resolução $\times$ largura** respeitando o ponto forte de cada recorte:

1. **Escalonamento Composto de Resolução (`MODO_EXPERIMENTO_RESOLUCAO = "compound_adaptativo"`):**
   - **$\alpha = 0.35$ @ `192×192` (`Grid 6×6 = 36 células`, $+44\%$ de área espacial):** Como possui gargalos estreitos (poucos canais por camada), elevar a entrada para $192\times 192$ amplia a nitidez de pústulas iniciais de ferrugem e estrias sem adicionar **1 único byte** ao `.tflite` (`~624 KB`), mantendo a inferência estimada no ESP32-S3 abaixo do teto de tempo real (`~386 ms < 500 ms`).
   - **$\alpha = 0.50$ @ `192×192` (`Grid 6×6 = 36 células` + $1,7\times$ parâmetros):** Combina o mapa $6\times 6$ com ~700k parâmetros (`~965 KB` Flash, `~27,3 KB` Scratch SRAM), operando na zona viável de voo pairado (*Pause-and-Classify*, `~634 ms`).
   - **$\alpha = 0.75$ @ `224×224` (`Grid 7×7 = 49 células`, $+96\%$ de área espacial + $3,4\times$ parâmetros):** Alimenta os 1,38M parâmetros na resolução canônica do ImageNet ($224\times 224$), quase dobrando a resolução espacial no topo da rede para mitigar *overfitting* e extrair a acurácia máxima de referência do backbone.
   *(Nota: A constante `MODO_EXPERIMENTO_RESOLUCAO` também permite alternar rapidamente para `"alta_resolucao_224"`, `"resolucao_192"` ou `"baseline_160"` para comparações cruzadas).*

2. **Regularização Espacial 2D (`SpatialDropout2D`) com Custo Zero no Firmware:**
   - Inserida antes do `GlobalAveragePooling2D` (`0.05` / `0.12` / `0.20`), força independência entre mapas de características adjacentes durante o treinamento Float32 e é completamente eliminada na exportação TFLite Int8, preservando **100% dos exatos 6 operadores suportados pelo `MicroMutableOpResolver<6>`** do ESP32-S3 (`CONV_2D`, `DEPTHWISE_CONV_2D`, `ADD`, `MEAN`, `FULLY_CONNECTED`, `SOFTMAX`).

3. **Pipeline de Treinamento em 4 Estágios com *Clean Cooldown* Pré-QAT:**
   - **Fase 1 (*Head Warmup*):** Alinhamento inicial do classificador com backbone 100% congelado.
   - **Fase 2A (*Fine-Tuning* Estrutural c/ Regularização Forte):** Aumento geométrico + ruído Gaussiano de sensor + MixUp + congelamento inicial seletivo (`0%` / `30%` / `50%`) mantendo `BatchNormalization` congelada.
   - **Fase 2B (*Descongelamento Progressivo + Clean Cooldown*):** Reduz a fração congelada (`0%` / `10%` / `25%`), reduz a taxa de aprendizado e **desativa MixUp e ruído artificial**, calibrando os *logits* na distribuição real antes da quantização Int8.
   - **Fase 3 (*QAT Int8 Puro*):** Reabilita `trainable=True` para *BatchNorm Folding* no `tfmot`, refina pesos/escalas `FakeQuant` e exporta `.tflite`, `.h` e `training_reports_v10.md`.

## 0. Instalação e Configuração do Ambiente Reprodutível

In [ ]:
!pip install kagglehub tensorflow-model-optimization scikit-learn seaborn --quiet

In [ ]:
# Célula de imports — forçar uso consistente do tf.keras (Legacy Keras 2 para compatibilidade total com TFMOT)
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gc
import time
import math
import shutil
import hashlib
from datetime import datetime
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Tudo via tf.keras — evita conflito com Keras 3 standalone no Google Colab
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    matthews_corrcoef
)
import tensorflow_model_optimization as tfmot

# Configuração de sementes globais para reprodutibilidade científica
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Auditoria Criptográfica MD5, Descontaminação Cruzada e Split Estratificado (80/20)

> **Rigor Metodológico (Prevenção de *Data Leakage*):**
> Conforme documentado em [`01_AUDITORIA_MD5_DATASET_E_TREINAMENTO_V08.md`](docs/02_engenharia_de_dados_e_modelos/01_AUDITORIA_MD5_DATASET_E_TREINAMENTO_V08.md), o dataset bruto (6.748 imagens em 11 classes) possui **747 duplicatas internas** e **306 pares de colisões cruzadas MD5** entre pastas distintas.
> Esta etapa repete exatamente a descontaminação determinística (`SEED=42`), retendo **5.389 imagens únicas e limpas** (4.313 treino / 1.076 validação estratificada) para garantir comparação justa com `v08` e `v09`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Ambiente fora do Google Colab; ignorando drive.mount().")

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# Caminho da pasta do dataset (Google Drive no Colab ou caminho local alternativo)
CAMINHO_DATASET_DRIVE = "/content/drive/MyDrive/tcc/datasets/Sugarcane leaf image dataset"
CAMINHO_DATASET_LOCAL = "/home/victor/Documents/datasets/Sugarcane leaf image dataset-20261106T004142Z-1-001/Sugarcane leaf image dataset"
CAMINHO_DATASET_ORIGEM = CAMINHO_DATASET_DRIVE if os.path.isdir(CAMINHO_DATASET_DRIVE) else CAMINHO_DATASET_LOCAL
caminho_dataset_split = "/content/sugarcane_split_v10" if os.path.isdir("/content") else "./sugarcane_split_v10"

# Flags de curadoria (manter True para eliminar data leakage e rótulos contraditórios)
REMOVER_DUPLICATAS_INTERNAS = True
REMOVER_COLISOES_CRUZADAS = True
VAL_SPLIT = 0.20

def tem_imagens_diretas(pasta):
    try:
        return any(
            f.lower().endswith(IMG_EXTS) and os.path.isfile(os.path.join(pasta, f))
            for f in os.listdir(pasta)
        )
    except (FileNotFoundError, NotADirectoryError):
        return False

def descobrir_classes(raiz):
    classes = {}
    for item in sorted(os.listdir(raiz)):
        caminho_item = os.path.join(raiz, item)
        if not os.path.isdir(caminho_item):
            continue
        if tem_imagens_diretas(caminho_item):
            classes[item] = caminho_item
        else:
            for sub in sorted(os.listdir(caminho_item)):
                caminho_sub = os.path.join(caminho_item, sub)
                if os.path.isdir(caminho_sub) and tem_imagens_diretas(caminho_sub):
                    classes[sub] = caminho_sub
    return classes

def calcular_md5(caminho_arquivo):
    with open(caminho_arquivo, "rb") as fp:
        return hashlib.md5(fp.read()).hexdigest()

def construir_dataset_limpo_estratificado(raiz_origem, raiz_destino, modo="symlink"):
    if not os.path.isdir(raiz_origem):
        raise FileNotFoundError(f"Pasta de origem não encontrada: {raiz_origem}")

    classes = descobrir_classes(raiz_origem)
    if not classes:
        raise RuntimeError(f"Nenhuma classe encontrada em: {raiz_origem}")

    if os.path.exists(raiz_destino):
        shutil.rmtree(raiz_destino)

    print("🔍 Etapa 1/2: Auditando hashes MD5 de todas as imagens do dataset...")
    mapa_hashes = defaultdict(list)
    arquivos_por_classe = {}

    for nome_classe, caminho_origem in sorted(classes.items()):
        imgs = sorted([f for f in os.listdir(caminho_origem) if f.lower().endswith(IMG_EXTS)])
        arquivos_por_classe[nome_classe] = []
        for f in imgs:
            src = os.path.join(caminho_origem, f)
            h = calcular_md5(src)
            mapa_hashes[h].append((nome_classe, f, src))
            arquivos_por_classe[nome_classe].append((f, src, h))

    hashes_contaminados = {
        h for h, itens in mapa_hashes.items()
        if len(set(x[0] for x in itens)) > 1
    }

    total_brutos = sum(len(v) for v in arquivos_por_classe.values())
    print(f"   - Total de arquivos brutos : {total_brutos}")
    print(f"   - Hashes MD5 únicos totais : {len(mapa_hashes)}")
    print(f"   - Hashes c/ colisão cruzada: {len(hashes_contaminados)} pares entre classes distintas\n")

    print("📦 Etapa 2/2: Construindo split estratificado limpo (80% Treino / 20% Validação)...")
    print(f"{'Classe':<18} | {'Bruto':>5} | {'Dups Int.':>9} | {'Colisão Cruz.':>13} | {'Limpos':>6} | {'Treino':>6} | {'Val':>5}")
    print("-" * 80)

    rng = np.random.default_rng(SEED)
    contagem_treino = {}
    contagem_val = {}
    resumo_auditoria = {
        "total_brutos": total_brutos,
        "hashes_unicos": len(mapa_hashes),
        "pares_colisao": len(hashes_contaminados),
        "por_classe": {}
    }

    for nome_classe in sorted(classes.keys()):
        vistos_na_classe = set()
        limpos = []
        dups_internas = 0
        colisoes_cruzadas = 0

        for f, src, h in arquivos_por_classe[nome_classe]:
            if REMOVER_DUPLICATAS_INTERNAS and h in vistos_na_classe:
                dups_internas += 1
                continue
            vistos_na_classe.add(h)

            if REMOVER_COLISOES_CRUZADAS and h in hashes_contaminados:
                colisoes_cruzadas += 1
                continue

            limpos.append((f, src))

        indices = np.arange(len(limpos))
        rng.shuffle(indices)
        n_val = max(1, int(round(len(limpos) * VAL_SPLIT)))
        val_idx = set(indices[:n_val])

        pasta_tr = os.path.join(raiz_destino, "train", nome_classe)
        pasta_va = os.path.join(raiz_destino, "val", nome_classe)
        os.makedirs(pasta_tr, exist_ok=True)
        os.makedirs(pasta_va, exist_ok=True)

        n_tr_count = 0
        n_va_count = 0
        for idx, (f, src) in enumerate(limpos):
            if idx in val_idx:
                dst = os.path.join(pasta_va, f)
                n_va_count += 1
            else:
                dst = os.path.join(pasta_tr, f)
                n_tr_count += 1

            if not os.path.exists(dst):
                if modo == "symlink":
                    os.symlink(os.path.abspath(src), dst)
                else:
                    shutil.copy2(src, dst)

        contagem_treino[nome_classe] = n_tr_count
        contagem_val[nome_classe] = n_va_count
        resumo_auditoria["por_classe"][nome_classe] = {
            "bruto": len(arquivos_por_classe[nome_classe]),
            "dups_internas": dups_internas,
            "colisoes_cruzadas": colisoes_cruzadas,
            "limpos": len(limpos),
            "treino": n_tr_count,
            "val": n_va_count,
        }
        print(f"{nome_classe:<18} | {len(arquivos_por_classe[nome_classe]):>5} | {dups_internas:>9} | {colisoes_cruzadas:>13} | {len(limpos):>6} | {n_tr_count:>6} | {n_va_count:>5}")

    print("-" * 80)
    print(f"{'TOTAL':<18} | {total_brutos:>5} | {'-':>9} | {'-':>13} | {sum(contagem_treino.values()) + sum(contagem_val.values()):>6} | {sum(contagem_treino.values()):>6} | {sum(contagem_val.values()):>5}")
    print(f"\n✅ Dataset limpo e estratificado pronto em: {raiz_destino}")
    return os.path.join(raiz_destino, "train"), os.path.join(raiz_destino, "val"), contagem_treino, contagem_val, resumo_auditoria

pasta_train, pasta_val, contagem_treino_limpo, contagem_val_limpo, auditoria_dataset_md5 = construir_dataset_limpo_estratificado(
    CAMINHO_DATASET_ORIGEM, caminho_dataset_split, modo="symlink"
)

## 2. Pipeline Multi-Resolução (`Box Averaging`) e Perfis Especializados por Recorte ($\alpha$)

### Estratégia Composta da Versão v10: Resolução $\times$ Capacidade $\times$ Regularização Espacial
Para extrair o potencial máximo de cada multiplicador de largura $\alpha \in \{0.35, 0.50, 0.75\}$ sem perder a paridade `interpolation="area"` (`BoxAvg` do firmware `JPEGDraw`), cada recorte recebe uma resolução espacial e um regime de regularização sob medida:

| Parâmetro / Estratégia | MobileNetV2 $\alpha = 0.35$ | MobileNetV2 $\alpha = 0.50$ | MobileNetV2 $\alpha = 0.75$ | Propósito de Engenharia |
| :--- | :---: | :---: | :---: | :--- |
| **Resolução Alvo (`compound_adaptativo`)** | **`192 × 192`** ($+44\%$ área) | **`192 × 192`** ($+44\%$ área) | **`224 × 224`** ($+96\%$ área) | Elimina *resolution starvation* ($5\times5 \rightarrow 6\times6 / 7\times7$) |
| **Grid Espacial Pré-GAP** | $6 \times 6$ (36 células) | $6 \times 6$ (36 células) | $7 \times 7$ (49 células) | Preserva lesões apicais e pústulas pequenas |
| **`SpatialDropout2D` (Pré-GAP)** | `0.05` | `0.12` | `0.20` | Evita co-adaptação espacial 2D (custo zero no `.tflite`) |
| **`Dropout` 1D + `L2`** | `0.30` / `1e-5` | `0.40` / `5e-5` | `0.50` / `1.5e-4` | Contém saturação no classificador `Dense` |
| **Congelamento Fase 2A $\rightarrow$ Fase 2B** | `0%` $\rightarrow$ `0%` | `30%` $\rightarrow$ `10%` | `50%` $\rightarrow$ `25%` | Descongelamento progressivo profundo + *Clean Cooldown* |
| **Ruído de Sensor ($\sigma$) / MixUp ($\beta$)** | `0.0` / `0.0` | `0.015` / `0.15` | `0.025` / `0.25` | Desativados na Fase 2B e Fase 3 (QAT) |
| **Tamanho Flash `.tflite` Estimado** | **~624 KB** | **~965 KB** | **~1.720 KB** | Mesma pegada Flash da `v09` (pesos independem da resolução) |

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Seletor do experimento de resolução por recorte α:
#   - "compound_adaptativo" : α=0.35 -> 192x192 | α=0.50 -> 192x192 | α=0.75 -> 224x224 (Padrão v10)
#   - "alta_resolucao_224"  : todos em 224x224
#   - "resolucao_192"       : todos em 192x192
#   - "baseline_160"        : todos em 160x160 (referência v09)
MODO_EXPERIMENTO_RESOLUCAO = "compound_adaptativo"

MAPAS_RESOLUCAO = {
    "compound_adaptativo": {0.35: (192, 192), 0.50: (192, 192), 0.75: (224, 224)},
    "alta_resolucao_224":  {0.35: (224, 224), 0.50: (224, 224), 0.75: (224, 224)},
    "resolucao_192":       {0.35: (192, 192), 0.50: (192, 192), 0.75: (192, 192)},
    "baseline_160":        {0.35: (160, 160), 0.50: (160, 160), 0.75: (160, 160)},
}

if MODO_EXPERIMENTO_RESOLUCAO not in MAPAS_RESOLUCAO:
    raise ValueError(f"Modo inválido: {MODO_EXPERIMENTO_RESOLUCAO}")

RESOLUCOES_ATIVAS = MAPAS_RESOLUCAO[MODO_EXPERIMENTO_RESOLUCAO]

# 1. Classes e pesos de classe determinísticos sobre o dataset descontaminado (p^0.75)
class_names = sorted(contagem_treino_limpo.keys())
NUM_CLASSES = len(class_names)
print(f"Classes ({NUM_CLASSES}): {class_names}")

class_counts = [contagem_treino_limpo[nome] for nome in class_names]
total_amostras = sum(class_counts)
pesos_brutos = [total_amostras / (NUM_CLASSES * c) for c in class_counts]
class_weights_dict = {i: float(p ** 0.75) for i, p in enumerate(pesos_brutos)}

print("\nPesos de classe aplicados (p^0.75 sobre dataset descontaminado):")
for i, nome in enumerate(class_names):
    print(f"  - {nome:<18}: {class_counts[i]:>4} imgs treino | {contagem_val_limpo[nome]:>3} imgs val | peso = {class_weights_dict[i]:.2f}")

# 2. Normalização padrão Int8: converte [0.0, 255.0] para [-1.0, 1.0] fora do grafo Keras
def normalizar_padrao_int8(img, label):
    img = (tf.cast(img, tf.float32) / 127.5) - 1.0
    return img, label

# Cache multi-resolução para evitar recarregar/reprocessar quando duas escalas usam o mesmo img_size
CACHE_PIPELINES_RESOLUCAO = {}

def obter_pipelines_por_resolucao(img_size):
    """Carrega train_ds_cached e val_ds_norm na resolução solicitada com interpolation='area'."""
    img_size = tuple(img_size)
    if img_size in CACHE_PIPELINES_RESOLUCAO:
        return CACHE_PIPELINES_RESOLUCAO[img_size]

    print(f"\n📐 Preparando pipeline tf.data para resolução {img_size[0]}x{img_size[1]} (interpolation='area')...")
    train_ds_raw = tf.keras.utils.image_dataset_from_directory(
        pasta_train,
        seed=SEED,
        image_size=img_size,
        interpolation="area",
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    val_ds_raw = tf.keras.utils.image_dataset_from_directory(
        pasta_val,
        seed=SEED,
        image_size=img_size,
        interpolation="area",
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    assert train_ds_raw.class_names == class_names, "Divergência na ordenação de classes!"

    val_ds_norm = (
        val_ds_raw
        .map(normalizar_padrao_int8, num_parallel_calls=AUTOTUNE)
        .cache()
        .prefetch(AUTOTUNE)
    )
    train_ds_cached = train_ds_raw.cache()

    for imgs, _ in val_ds_norm.take(1):
        print(
            f"   ✅ Cache {img_size[0]}x{img_size[1]} validado! "
            f"Range: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}] | Shape: {imgs.shape}"
        )

    CACHE_PIPELINES_RESOLUCAO[img_size] = (train_ds_cached, val_ds_norm)
    return train_ds_cached, val_ds_norm

# 3. Perfis Especializados por Recorte (alpha) + Escalonamento Composto de Resolução
PERFIS_TREINAMENTO = {
    0.35: {
        "nome_curto": f"mbnet_a035_r{RESOLUCOES_ATIVAS[0.35][0]}",
        "img_size": RESOLUCOES_ATIVAS[0.35],
        "grid_final": f"{RESOLUCOES_ATIVAS[0.35][0]//32}x{RESOLUCOES_ATIVAS[0.35][1]//32}",
        "titulo": f"MobileNetV2 (α=0.35 @ {RESOLUCOES_ATIVAS[0.35][0]}×{RESOLUCOES_ATIVAS[0.35][1]} — Leve / Tempo Real)",
        "rot_factor": 0.05,
        "zoom_range": (-0.08, 0.05),
        "contrast_factor": 0.12,
        "translation_factor": 0.0,
        "noise_std": 0.0,
        "mixup_alpha": 0.0,
        "mixup_prob": 0.0,
        "spatial_dropout_rate": 0.05,
        "dropout_rate": 0.30,
        "l2_weight_decay": 1e-5,
        "label_smoothing": 0.02,
        "freeze_ratio_phase2a": 0.00,
        "freeze_ratio_phase2b": 0.00,
        "lr_phase1": 1.0e-3,
        "lr_phase2a": 5.0e-5,
        "lr_phase2b": 1.5e-5,
        "lr_qat": 1.0e-5,
        "epochs_phase1": 15,
        "epochs_phase2a": 40,
        "epochs_phase2b": 10,
        "epochs_qat": 25,
        "patience_phase2a": 10,
        "patience_phase2b": 4,
    },
    0.50: {
        "nome_curto": f"mbnet_a050_r{RESOLUCOES_ATIVAS[0.50][0]}",
        "img_size": RESOLUCOES_ATIVAS[0.50],
        "grid_final": f"{RESOLUCOES_ATIVAS[0.50][0]//32}x{RESOLUCOES_ATIVAS[0.50][1]//32}",
        "titulo": f"MobileNetV2 (α=0.50 @ {RESOLUCOES_ATIVAS[0.50][0]}×{RESOLUCOES_ATIVAS[0.50][1]} — Equilibrado / Voo Pairado)",
        "rot_factor": 0.065,
        "zoom_range": (-0.10, 0.06),
        "contrast_factor": 0.15,
        "translation_factor": 0.03,
        "noise_std": 0.015,
        "mixup_alpha": 0.15,
        "mixup_prob": 0.30,
        "spatial_dropout_rate": 0.12,
        "dropout_rate": 0.40,
        "l2_weight_decay": 5e-5,
        "label_smoothing": 0.04,
        "freeze_ratio_phase2a": 0.30,
        "freeze_ratio_phase2b": 0.10,
        "lr_phase1": 1.0e-3,
        "lr_phase2a": 3.5e-5,
        "lr_phase2b": 1.2e-5,
        "lr_qat": 1.0e-5,
        "epochs_phase1": 15,
        "epochs_phase2a": 40,
        "epochs_phase2b": 12,
        "epochs_qat": 25,
        "patience_phase2a": 9,
        "patience_phase2b": 4,
    },
    0.75: {
        "nome_curto": f"mbnet_a075_r{RESOLUCOES_ATIVAS[0.75][0]}",
        "img_size": RESOLUCOES_ATIVAS[0.75],
        "grid_final": f"{RESOLUCOES_ATIVAS[0.75][0]//32}x{RESOLUCOES_ATIVAS[0.75][1]//32}",
        "titulo": f"MobileNetV2 (α=0.75 @ {RESOLUCOES_ATIVAS[0.75][0]}×{RESOLUCOES_ATIVAS[0.75][1]} — Máxima Capacidade)",
        "rot_factor": 0.08,
        "zoom_range": (-0.12, 0.08),
        "contrast_factor": 0.18,
        "translation_factor": 0.05,
        "noise_std": 0.025,
        "mixup_alpha": 0.25,
        "mixup_prob": 0.45,
        "spatial_dropout_rate": 0.20,
        "dropout_rate": 0.50,
        "l2_weight_decay": 1.5e-4,
        "label_smoothing": 0.06,
        "freeze_ratio_phase2a": 0.50,
        "freeze_ratio_phase2b": 0.25,
        "lr_phase1": 8.0e-4,
        "lr_phase2a": 2.0e-5,
        "lr_phase2b": 8.0e-6,
        "lr_qat": 8.0e-6,
        "epochs_phase1": 15,
        "epochs_phase2a": 35,
        "epochs_phase2b": 12,
        "epochs_qat": 22,
        "patience_phase2a": 8,
        "patience_phase2b": 4,
    },
}

def construir_pipeline_treino_por_escala(alpha, modo_limpo=False):
    """
    Constrói pipeline tf.data específico para (alpha, img_size).
    Quando modo_limpo=True (Fase 2B Clean Cooldown e Fase 3 QAT), desativa MixUp e ruído artificial
    para alinhar fronteiras de decisão e faixas FakeQuant na distribuição real.
    """
    cfg = PERFIS_TREINAMENTO[alpha]
    train_ds_cached, _ = obter_pipelines_por_resolucao(cfg["img_size"])

    camadas_aug = [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(cfg["rot_factor"], fill_mode="reflect"),
        layers.RandomZoom(cfg["zoom_range"], fill_mode="reflect"),
        layers.RandomContrast(cfg["contrast_factor"]),
    ]
    if cfg["translation_factor"] > 0:
        camadas_aug.append(
            layers.RandomTranslation(
                cfg["translation_factor"], cfg["translation_factor"], fill_mode="reflect"
            )
        )
    aug_seq = tf.keras.Sequential(camadas_aug, name=f"aug_a{int(alpha*100):03d}")

    noise_std = 0.0 if modo_limpo else float(cfg["noise_std"])
    mix_alpha = 0.0 if modo_limpo else float(cfg["mixup_alpha"])
    mix_prob = 0.0 if modo_limpo else float(cfg["mixup_prob"])

    def aplicar_aug_e_regularizacao_batch(imgs, lbls):
        imgs_aug = aug_seq(imgs, training=True)
        imgs_norm = (tf.cast(imgs_aug, tf.float32) / 127.5) - 1.0

        if noise_std > 0.0:
            ruido = tf.random.normal(tf.shape(imgs_norm), mean=0.0, stddev=noise_std, dtype=tf.float32)
            imgs_norm = tf.clip_by_value(imgs_norm + ruido, -1.0, 1.0)

        if mix_alpha > 0.0 and mix_prob > 0.0:
            aplicar = tf.random.uniform([], 0.0, 1.0) < mix_prob
            def fazer_mixup():
                batch_sz = tf.shape(imgs_norm)[0]
                indices = tf.random.shuffle(tf.range(batch_sz))
                g1 = tf.random.gamma([batch_sz], alpha=mix_alpha, beta=1.0)
                g2 = tf.random.gamma([batch_sz], alpha=mix_alpha, beta=1.0)
                lam = g1 / (g1 + g2 + 1e-7)
                lam = tf.maximum(lam, 1.0 - lam)
                lam_img = tf.reshape(lam, [batch_sz, 1, 1, 1])
                lam_lbl = tf.reshape(lam, [batch_sz, 1])
                imgs_mix = lam_img * imgs_norm + (1.0 - lam_img) * tf.gather(imgs_norm, indices)
                lbls_mix = lam_lbl * lbls + (1.0 - lam_lbl) * tf.gather(lbls, indices)
                return imgs_mix, lbls_mix

            imgs_norm, lbls = tf.cond(aplicar, fazer_mixup, lambda: (imgs_norm, lbls))

        return imgs_norm, lbls

    ds = (
        train_ds_cached
        .shuffle(1000, seed=SEED)
        .map(aplicar_aug_e_regularizacao_batch, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )
    return ds

def criar_loss_adaptativa(label_smoothing=0.02):
    """Cria CategoricalFocalCrossentropy (gamma=2.0) com label_smoothing adaptado à escala."""
    if hasattr(keras.losses, "CategoricalFocalCrossentropy"):
        return keras.losses.CategoricalFocalCrossentropy(gamma=2.0, label_smoothing=label_smoothing)
    return keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)

print(f"\n✅ Modo de experimento selecionado: {MODO_EXPERIMENTO_RESOLUCAO}")
for a, c in PERFIS_TREINAMENTO.items():
    print(f"   - α={a:.2f}: entrada {c['img_size'][0]}x{c['img_size'][1]} (grid {c['grid_final']}) | SpatialDropout={c['spatial_dropout_rate']} | Dropout={c['dropout_rate']}")

---
## 3. Construtor Modular Flat-Graph (`SpatialDropout2D`) e Pipeline de Treinamento em 4 Estágios

### Arquitetura do Protocolo de Treinamento em 4 Estágios (`v10`)
1. **Fase 1 — *Head Warmup* (Backbone 100% Congelado):** Alinha o classificador `Dense` inicial com os mapas pré-treinados do ImageNet.
2. **Fase 2A — *Fine-Tuning* Estrutural Regulado:** Descongela a porção superior do backbone (`0%` / `30%` / `50%` iniciais congelados) mantendo `BatchNormalization` congelada, com MixUp, ruído de sensor e `SpatialDropout2D`.
3. **Fase 2B — Descongelamento Progressivo Profundo + *Clean Cooldown* Pré-QAT:** Libera uma faixa mais profunda do backbone (`0%` / `10%` / `25%` congelados), reduz o *learning rate* e **desliga MixUp e ruído sintético** para estabilizar as fronteiras de decisão antes da quantização.
4. **Fase 3 — *Quantization-Aware Training* (QAT Int8):** Reabilita `trainable=True` para o *BatchNorm Folding* no `tfmot.quantize_model()` e refina os nós `FakeQuant` de 8 bits.

In [ ]:
def build_mobilenetv2_flat(num_classes, alpha=0.35, incluir_spatial_dropout=True):
    """
    Constrói a MobileNetV2 com grafo flat (input_tensor=inputs) na resolução específica
    do recorte (cfg['img_size']) e SpatialDropout2D pré-GAP (custo zero em inferência TFLite).
    """
    cfg = PERFIS_TREINAMENTO[alpha]
    img_shape = (cfg["img_size"][0], cfg["img_size"][1], 3)
    inputs = keras.Input(shape=img_shape, name="input_image")

    base_model = keras.applications.MobileNetV2(
        input_tensor=inputs,
        alpha=alpha,
        include_top=False,
        weights='imagenet'
    )

    # Fase 1: Congela todo o extrator de características pré-treinado
    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    if incluir_spatial_dropout and cfg["spatial_dropout_rate"] > 0.0:
        x = layers.SpatialDropout2D(cfg["spatial_dropout_rate"], name="spatial_dropout")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(cfg["dropout_rate"], name="top_dropout")(x)
    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        kernel_regularizer=regularizers.l2(cfg["l2_weight_decay"]),
        name="predictions"
    )(x)

    nome_modelo = f"mobilenetv2_a{int(alpha*100):03d}_r{cfg['img_size'][0]}"
    model = keras.Model(inputs=inputs, outputs=outputs, name=nome_modelo)
    return model

def configurar_descongelamento_seletivo(model, freeze_ratio, estagio="Fase 2A"):
    """
    Descongela seletivamente o backbone conforme freeze_ratio, mantendo TODAS as camadas
    BatchNormalization congeladas para proteger as médias móveis pré-treinadas.
    """
    head_names = {"spatial_dropout", "gap", "top_dropout", "predictions"}
    backbone_indices = [i for i, l in enumerate(model.layers) if l.name not in head_names]
    n_total_backbone = len(backbone_indices)
    n_congeladas = int(round(n_total_backbone * freeze_ratio))
    indice_corte = backbone_indices[n_congeladas] if n_congeladas < n_total_backbone else len(model.layers)

    n_treinaveis = 0
    for idx, layer in enumerate(model.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        elif idx < indice_corte and layer.name not in head_names:
            layer.trainable = False
        else:
            layer.trainable = True
            if layer.weights:
                n_treinaveis += 1

    print(
        f"   🔒 Controle Anti-Overfitting ({estagio}): "
        f"{n_congeladas}/{n_total_backbone} camadas iniciais congeladas ({freeze_ratio*100:.0f}%) "
        f"| BatchNorm congelado | {n_treinaveis} camadas com pesos ativas."
    )

def preparar_modelo_para_qat(model_f32, alpha):
    """
    Reabilita trainable=True para BatchNorm Folding e aplica tfmot.quantize_model().
    Se alguma versão do TFMOT recusar SpatialDropout2D, remove a camada sem pesos
    copiando 100% dos pesos treinados antes de aplicar o QAT.
    """
    for layer in model_f32.layers:
        layer.trainable = True

    try:
        return tfmot.quantization.keras.quantize_model(model_f32)
    except Exception as exc:
        print(f"   ⚠️ Fallback QAT acionado ({exc}): reconstruindo grafo limpo sem SpatialDropout2D preservando pesos...")
        model_limpo = build_mobilenetv2_flat(NUM_CLASSES, alpha=alpha, incluir_spatial_dropout=False)
        for layer in model_limpo.layers:
            layer.trainable = True
            if layer.weights:
                layer.set_weights(model_f32.get_layer(layer.name).get_weights())
        return tfmot.quantization.keras.quantize_model(model_limpo)

def treinar_pipeline_completo_escala(alpha):
    """Executa Fase 1 (Warmup), Fase 2A (Fine-Tune), Fase 2B (Clean Cooldown) e Fase 3 (QAT)."""
    cfg = PERFIS_TREINAMENTO[alpha]
    print("=" * 104)
    print(f"🚀 INICIANDO TREINAMENTO v10: {cfg['titulo']}")
    print(
        f"   Resolução: {cfg['img_size'][0]}x{cfg['img_size'][1]} (Grid {cfg['grid_final']}) | "
        f"SpatialDropout={cfg['spatial_dropout_rate']} | Dropout={cfg['dropout_rate']} | "
        f"L2={cfg['l2_weight_decay']} | Smooth={cfg['label_smoothing']} | "
        f"Noise={cfg['noise_std']} | MixUp={cfg['mixup_alpha']}"
    )
    print("=" * 104)

    _, val_ds_norm_escala = obter_pipelines_por_resolucao(cfg["img_size"])
    train_ds_escala = construir_pipeline_treino_por_escala(alpha, modo_limpo=False)
    train_ds_limpo = construir_pipeline_treino_por_escala(alpha, modo_limpo=True)
    loss_fn = criar_loss_adaptativa(label_smoothing=cfg["label_smoothing"])

    tempo_fases = {}
    t_inicio_total = time.time()

    # 1. Fase 1 (Head Warmup)
    model_f32 = build_mobilenetv2_flat(NUM_CLASSES, alpha=alpha, incluir_spatial_dropout=True)
    model_f32.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_phase1"]),
        loss=loss_fn,
        metrics=['accuracy']
    )
    print(f"\n[FASE 1/4] Head Warmup por até {cfg['epochs_phase1']} épocas (lr={cfg['lr_phase1']})...")
    t0 = time.time()
    hist_p1 = model_f32.fit(
        train_ds_escala,
        epochs=cfg["epochs_phase1"],
        validation_data=val_ds_norm_escala,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
            )
        ]
    )
    tempo_fases["fase1_min"] = (time.time() - t0) / 60.0

    # 2A. Fase 2A (Fine-Tuning Estrutural c/ Regularização Completa)
    print(f"\n[FASE 2A/4] Fine-Tuning Estrutural por até {cfg['epochs_phase2a']} épocas (lr={cfg['lr_phase2a']})...")
    configurar_descongelamento_seletivo(model_f32, cfg["freeze_ratio_phase2a"], estagio="Fase 2A")
    model_f32.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_phase2a"], clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy']
    )
    t0 = time.time()
    hist_p2a = model_f32.fit(
        train_ds_escala,
        epochs=cfg["epochs_phase2a"],
        validation_data=val_ds_norm_escala,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=cfg["patience_phase2a"], restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
            )
        ]
    )
    tempo_fases["fase2a_min"] = (time.time() - t0) / 60.0

    # 2B. Fase 2B (Descongelamento Progressivo + Clean Cooldown Pré-QAT)
    print(
        f"\n[FASE 2B/4] Clean Cooldown + Descongelamento Profundo por até {cfg['epochs_phase2b']} épocas "
        f"(lr={cfg['lr_phase2b']}, sem MixUp/Ruído)..."
    )
    configurar_descongelamento_seletivo(model_f32, cfg["freeze_ratio_phase2b"], estagio="Fase 2B Clean Cooldown")
    model_f32.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_phase2b"], clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy']
    )
    t0 = time.time()
    hist_p2b = model_f32.fit(
        train_ds_limpo,
        epochs=cfg["epochs_phase2b"],
        validation_data=val_ds_norm_escala,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=cfg["patience_phase2b"], restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
            )
        ]
    )
    tempo_fases["fase2b_min"] = (time.time() - t0) / 60.0

    val_loss_f32, val_acc_f32 = model_f32.evaluate(val_ds_norm_escala, verbose=0)
    print(f"\n✅ [{cfg['nome_curto']}] Float32 finalizado! Val Accuracy: {val_acc_f32*100:.2f}% | Val Loss: {val_loss_f32:.4f}")

    # 3. Fase 3 (Quantization-Aware Training - QAT Int8)
    print(f"\n[FASE 3/4] Aplicando Quantization-Aware Training (QAT Int8) por até {cfg['epochs_qat']} épocas (lr={cfg['lr_qat']})...")
    t0 = time.time()
    model_qat = preparar_modelo_para_qat(model_f32, alpha)
    model_qat.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_qat"], clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy']
    )
    hist_qat = model_qat.fit(
        train_ds_limpo,
        epochs=cfg["epochs_qat"],
        validation_data=val_ds_norm_escala,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=6, restore_best_weights=True, start_from_epoch=4, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
            )
        ]
    )
    tempo_fases["fase3_qat_min"] = (time.time() - t0) / 60.0

    tempo_total_min = (time.time() - t_inicio_total) / 60.0
    val_loss_qat, val_acc_qat = model_qat.evaluate(val_ds_norm_escala, verbose=0)
    print(
        f"\n🏁 [{cfg['nome_curto']}] Pipeline completo em {tempo_total_min:.1f} min! "
        f"Acc Float32: {val_acc_f32*100:.2f}% -> Acc Pós-QAT: {val_acc_qat*100:.2f}% "
        f"(Δ: {(val_acc_qat - val_acc_f32)*100:+.2f}%)\n"
    )

    gc.collect()
    return {
        "alpha": alpha,
        "cfg": cfg,
        "model_f32": model_f32,
        "model_qat": model_qat,
        "hist_p1": hist_p1,
        "hist_p2a": hist_p2a,
        "hist_p2b": hist_p2b,
        "hist_qat": hist_qat,
        "tempo_treino_min": tempo_total_min,
        "tempo_fases": tempo_fases,
    }

### 3.1 Treinamento da Variante A — MobileNetV2 ($\alpha = 0.35$ — Leve / Alta Definição Espacial)

In [ ]:
resultados_treino = {}
resultados_treino[0.35] = treinar_pipeline_completo_escala(0.35)

### 3.2 Treinamento da Variante B — MobileNetV2 ($\alpha = 0.50$ — Equilíbrio Capacidade × Resolução)

In [ ]:
resultados_treino[0.50] = treinar_pipeline_completo_escala(0.50)

### 3.3 Treinamento da Variante C — MobileNetV2 ($\alpha = 0.75$ — Capacidade Máxima @ $224\times 224$)

In [ ]:
resultados_treino[0.75] = treinar_pipeline_completo_escala(0.75)

---
## 4. Conversão Full-Integer (Int8 Puro) e Auditoria Multi-Resolução para o ESP32-S3

> **Inspeção Física para Co-Design Hardware/Software no ESP32-S3:**
> Cada variante é convertida para `.tflite` (`TFLITE_BUILTINS_INT8`, entradas e saídas `int8` em `[-128, 127]`) usando `representative_data_gen` na **resolução exata do respectivo recorte** e auditada contra os limites físicos do ESP32-S3:
> 1. **Partição Flash LittleFS (`1.536 KB` / `1,5 MB`):** Os pesos dependem apenas de $\alpha$, portanto elevar a resolução ($160\rightarrow 192\rightarrow 224$) **não aumenta o tamanho em Flash**.
> 2. **Orçamento de *Tensor Arena* em PSRAM Octal (`800 KB` padrão firmware / expansível até `2.048 KB` na PSRAM de 8 MB):** Tensores iniciais de maior resolução exigem ajuste proporcional de `kTensorArenaSize` em `src/tflite_classifier.cpp` caso ultrapassem `800 KB`.
> 3. **Orçamento de *Scratch Buffer* SIMD `esp-nn` em SRAM Interna (`Limite rígido de 48 KB`):** Graças ao *strip-tiling* automático do `esp_nn_depthwise_conv_s8_esp32s3` (limitado a ~40 KB), o aumento de resolução espacial não explode o scratch de SRAM interna.
> 4. **Estimativa Calibrada de Latência e Energia no Silício ESP32-S3 (240 MHz):** Ancorada no benchmark empírico real do ESP32-S3 para `MobileNetV2 α=0.35 @ 160×160` (`30,2 MMACs`, `623,9 KB`, **`283,4 ms`** de inferência pura / **`336,1 ms`** via WebSocket), escalando proporcionalmente pelos MMACs da resolução adotada (`(H/160)²`) e tráfego PSRAM.
> 5. **Validação Estrita de Operadores:** Garante que `SpatialDropout2D` desapareceu do grafo e que 100% dos nós pertencem aos 6 operadores registrados em `MicroMutableOpResolver<6>`.

In [ ]:
# Constantes de Hardware do ESP32-S3 N8R16 (referência empírica α=0.35 @ 160x160)
ESP32S3_FLASH_BUDGET_KB = 1536.0       # Partição LittleFS /models de 1.5 MB
ESP32S3_ARENA_BUDGET_KB = 800.0        # Tensor Arena padrão atual em src/tflite_classifier.cpp
ESP32S3_ARENA_MAX_PSRAM_KB = 2048.0    # Teto seguro expansível na PSRAM Octal de 8 MB
ESP32S3_SRAM_SCRATCH_LIMIT_KB = 48.0   # Limite estrito de SRAM interna p/ scratch buffer esp-nn
ESP32S3_REF_A035_160_MMACS = 30.2      # MMACs reais da MobileNetV2 α=0.35 @ 160x160
ESP32S3_REF_A035_160_KB = 623.9        # Tamanho real .tflite da MobileNetV2 α=0.35 @ 160x160
ESP32S3_BASE_LATENCY_A035_MS = 283.4   # Latência pura medida no ESP32-S3 físico (α=0.35 @ 160x160)
ESP32S3_WS_LATENCY_A035_MS = 336.1     # Latência média c/ telemetria no benchmark de 6.748 imagens
ESP32S3_ACTIVE_POWER_WATTS = 0.72      # Potência média do ESP32-S3 Dual-Core @ 240 MHz + PSRAM Octal

OPS_FIRMWARE_PERMITIDOS = {
    'CONV_2D', 'DEPTHWISE_CONV_2D', 'ADD', 'MEAN', 'FULLY_CONNECTED', 'SOFTMAX', 'DELEGATE'
}

def criar_representative_data_gen(val_dataset):
    """Cria gerador de calibração na resolução específica do recorte avaliado."""
    def representative_data_gen():
        for images, _ in val_dataset:
            imgs_np = images.numpy()
            for i in range(len(imgs_np)):
                yield [np.expand_dims(imgs_np[i], axis=0).astype(np.float32)]
    return representative_data_gen

def calcular_macs_e_parametros_keras(keras_model):
    """Calcula o total exato de parâmetros e MACs (Multiply-Accumulate) do grafo na resolução real."""
    total_params = keras_model.count_params()
    total_macs = 0
    max_in_channels_conv = 0

    for layer in keras_model.layers:
        if isinstance(layer, layers.DepthwiseConv2D):
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            _, h_out, w_out, c_out = [int(d) if d is not None else 1 for d in out_shape]
            kh, kw = layer.kernel_size
            total_macs += h_out * w_out * c_out * kh * kw
        elif isinstance(layer, layers.Conv2D):
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            in_shape = getattr(layer, 'input_shape', None) or layer.input.shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if isinstance(in_shape, list):
                in_shape = in_shape[0]
            _, h_out, w_out, c_out = [int(d) if d is not None else 1 for d in out_shape]
            c_in = int(in_shape[-1])
            kh, kw = layer.kernel_size
            total_macs += h_out * w_out * c_in * c_out * kh * kw
            if c_in > max_in_channels_conv:
                max_in_channels_conv = c_in
        elif isinstance(layer, layers.Dense):
            in_shape = getattr(layer, 'input_shape', None) or layer.input.shape
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            c_in = int(in_shape[-1])
            c_out = int(out_shape[-1])
            total_macs += c_in * c_out

    return total_params, total_macs, max_in_channels_conv

def converter_e_auditar_tflite_esp32s3(alpha, registro_treino):
    """Converte o modelo QAT para TFLite Int8 puro e audita métricas de hardware do ESP32-S3."""
    cfg = registro_treino["cfg"]
    model_f32 = registro_treino["model_f32"]
    model_qat = registro_treino["model_qat"]
    output_path = f"model_{cfg['nome_curto']}_int8.tflite"
    _, val_ds_norm_escala = obter_pipelines_por_resolucao(cfg["img_size"])

    print(f"⚙️ Convertendo {cfg['nome_curto']} ({cfg['img_size'][0]}x{cfg['img_size'][1]}) para TFLite Full-Integer Int8 ({output_path})...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model_qat)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = criar_representative_data_gen(val_ds_norm_escala)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_bytes = converter.convert()
    with open(output_path, "wb") as f:
        f.write(tflite_bytes)

    kb_size = len(tflite_bytes) / 1024.0
    total_params, total_macs, max_in_ch = calcular_macs_e_parametros_keras(model_f32)
    mmacs = total_macs / 1e6

    interpreter = tf.lite.Interpreter(model_path=output_path)
    interpreter.allocate_tensors()
    tensor_details = interpreter.get_tensor_details()
    ops_details = interpreter._get_ops_details()

    ops_names = [op['op_name'] for op in ops_details]
    unique_ops = set(ops_names)
    tem_float_op = any(op in ('DEQUANTIZE', 'QUANTIZE', 'MUL', 'SUB') for op in ops_names)
    firmware_ops_ok = unique_ops.issubset(OPS_FIRMWARE_PERMITIDOS)

    activation_sizes = []
    for t in tensor_details:
        shape = t.get('shape', [])
        if len(shape) == 4 and shape[0] == 1:
            activation_sizes.append(int(np.prod(shape)))

    activation_sizes_sorted = sorted(activation_sizes, reverse=True)
    peak_pair_kb = sum(activation_sizes_sorted[:2]) / 1024.0 if len(activation_sizes_sorted) >= 2 else 0.0
    estimated_arena_kb = peak_pair_kb * 1.35 + (len(ops_details) * 1.15)
    esp_nn_scratch_kb = (24.0 * max_in_ch + 16384.0 + 64.0) / 1024.0

    # Modelo híbrido calibrado no benchmark real α=0.35 @ 160x160
    fator_macs = mmacs / ESP32S3_REF_A035_160_MMACS
    fator_psram = kb_size / ESP32S3_REF_A035_160_KB
    fator_escala_latencia = 0.82 * fator_macs + 0.18 * fator_psram
    lat_pure_ms = ESP32S3_BASE_LATENCY_A035_MS * fator_escala_latencia
    lat_ws_ms = ESP32S3_WS_LATENCY_A035_MS * fator_escala_latencia

    hw_info = {
        "tflite_path": output_path,
        "img_size": cfg["img_size"],
        "grid_final": cfg["grid_final"],
        "kb_size": kb_size,
        "flash_usage_pct": (kb_size / ESP32S3_FLASH_BUDGET_KB) * 100.0,
        "fits_flash": kb_size <= ESP32S3_FLASH_BUDGET_KB,
        "total_params": total_params,
        "mmacs": mmacs,
        "max_in_ch": max_in_ch,
        "peak_pair_kb": peak_pair_kb,
        "estimated_arena_kb": estimated_arena_kb,
        "fits_arena_800kb": estimated_arena_kb <= ESP32S3_ARENA_BUDGET_KB,
        "fits_arena_2mb": estimated_arena_kb <= ESP32S3_ARENA_MAX_PSRAM_KB,
        "esp_nn_scratch_kb": esp_nn_scratch_kb,
        "fits_sram_48kb": esp_nn_scratch_kb <= ESP32S3_SRAM_SCRATCH_LIMIT_KB,
        "num_ops": len(ops_details),
        "unique_ops": unique_ops,
        "firmware_ops_ok": firmware_ops_ok,
        "is_pure_int8": not tem_float_op,
        "esp32s3_latency_pure_ms": lat_pure_ms,
        "esp32s3_latency_ws_ms": lat_ws_ms,
        "esp32s3_fps": 1000.0 / lat_pure_ms,
        "energy_mj": (lat_pure_ms / 1000.0) * ESP32S3_ACTIVE_POWER_WATTS * 1000.0,
    }
    return hw_info

auditoria_hw = {}
for alpha in [0.35, 0.50, 0.75]:
    hw = converter_e_auditar_tflite_esp32s3(alpha, resultados_treino[alpha])
    auditoria_hw[alpha] = hw
    status_ops = "OK (6 ops C++)" if hw["firmware_ops_ok"] else f"ALERTA ops={hw['unique_ops']}"
    print(
        f"✅ [{resultados_treino[alpha]['cfg']['nome_curto']}] "
        f"Res: {hw['img_size'][0]}x{hw['img_size'][1]} ({hw['grid_final']}) | "
        f"Flash: {hw['kb_size']:.1f} KB ({hw['flash_usage_pct']:.1f}%) | "
        f"Params: {hw['total_params']/1e3:.1f}k | MMACs: {hw['mmacs']:.1f}M | "
        f"Arena Est.: {hw['estimated_arena_kb']:.0f} KB | "
        f"Scratch SRAM: {hw['esp_nn_scratch_kb']:.1f} KB | "
        f"Latência Est.: {hw['esp32s3_latency_pure_ms']:.0f} ms ({hw['esp32s3_fps']:.2f} FPS) | Ops: {status_ops}"
    )

---
## 5. Avaliação Científica Padronizada, Métricas Clínicas/Agronômicas e Estado da Arte (SOTA)

Avaliamos todas as variantes (Float32 e Int8 QAT) sobre o conjunto de validação estratificado limpo ($N = 1.076$ imagens), extraindo indicadores estatísticos exigidos em publicações de alto impacto (IEEE / Elsevier *Computers and Electronics in Agriculture*):
- **Acurácia Top-1 (%) e Top-2 (%):** A acurácia Top-2 indica a frequência com que a patologia correta está entre as 2 maiores probabilidades do *Softmax* (essencial para triagem assistida em campo quando há sintomas iniciais ou co-infecção).
- **F1-Score Macro (%), Precisão Macro (%) e Sensibilidade/Recall Macro (%):** Média não-ponderada entre as 9 classes, penalizando severamente qualquer negligência às classes minoritárias (*Yellow Leaf* e *smut*).
- **Kappa de Cohen ($\kappa$) e Coeficiente de Correlação de Matthews (MCC):** Métricas robustas contra desbalanceamento de classes ($\kappa, \text{MCC} > 0,90$ indicam concordância quase perfeita).
- **Erro de Calibração Esperado (*Expected Calibration Error* — ECE %):** Mede a fidelidade das probabilidades emitidas após a quantização Int8.
- **NetScore ($\Omega$, Wong 2019):** Índice composto de eficiência de arquiteturas neurais embarcadas ($\text{dB}$), definido por:
  $$\Omega = 20 \cdot \log_{10}\left(\frac{\text{Acc}(\%)^2}{\text{Params(M)}^{0.5} \cdot \text{MACs(M)}^{0.5}}\right)$$

In [ ]:
def calcular_ece(probs, gabarito, n_bins=10):
    """Calcula o Expected Calibration Error (ECE %) das probabilidades preditas."""
    confiancas = np.max(probs, axis=1)
    predicoes = np.argmax(probs, axis=1)
    acertos = (predicoes == gabarito).astype(np.float32)

    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(gabarito)
    for b_idx in range(n_bins):
        b_low, b_high = bin_boundaries[b_idx], bin_boundaries[b_idx + 1]
        mask = (confiancas > b_low) & (confiancas <= b_high)
        if np.any(mask):
            acc_bin = np.mean(acertos[mask])
            conf_bin = np.mean(confiancas[mask])
            ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)
    return float(ece * 100.0)

def calcular_metricas_completas(probs, gabarito):
    """Extrai métricas globais, por classe, Top-2, Kappa, MCC e calibração."""
    predicoes = np.argmax(probs, axis=1)
    top2_preds = np.argsort(probs, axis=1)[:, -2:]
    top2_hits = [g in top2_preds[i] for i, g in enumerate(gabarito)]

    cm = confusion_matrix(gabarito, predicoes)
    acc_por_classe = (cm.diagonal() / np.maximum(cm.sum(axis=1), 1)) * 100.0
    f1_por_classe = f1_score(gabarito, predicoes, average=None) * 100.0

    acc_top1 = float(np.mean(predicoes == gabarito) * 100.0)
    acc_top2 = float(np.mean(top2_hits) * 100.0)
    f1_mac = float(f1_score(gabarito, predicoes, average='macro') * 100.0)
    prec_mac = float(precision_score(gabarito, predicoes, average='macro', zero_division=0) * 100.0)
    rec_mac = float(recall_score(gabarito, predicoes, average='macro', zero_division=0) * 100.0)
    kappa = float(cohen_kappa_score(gabarito, predicoes) * 100.0)
    mcc = float(matthews_corrcoef(gabarito, predicoes) * 100.0)

    confiancas = np.max(probs, axis=1) * 100.0
    mask_corretos = (predicoes == gabarito)
    conf_corretos = float(np.mean(confiancas[mask_corretos])) if np.any(mask_corretos) else 0.0
    conf_errados = float(np.mean(confiancas[~mask_corretos])) if np.any(~mask_corretos) else 0.0
    ece = calcular_ece(probs, gabarito)

    return {
        "cm": cm,
        "acc_por_classe": acc_por_classe,
        "f1_por_classe": f1_por_classe,
        "acc_top1": acc_top1,
        "acc_top2": acc_top2,
        "f1_macro": f1_mac,
        "precision_macro": prec_mac,
        "recall_macro": rec_mac,
        "kappa": kappa,
        "mcc": mcc,
        "conf_corretos": conf_corretos,
        "conf_errados": conf_errados,
        "ece": ece,
        "predicoes": predicoes,
        "gabarito": gabarito,
        "probs": probs,
    }

def avaliar_keras_completo(model, val_dataset, desc):
    probs_list, gabarito_list = [], []
    for images, labels in val_dataset:
        preds = model.predict(images, verbose=0)
        probs_list.append(preds)
        gabarito_list.extend(np.argmax(labels.numpy(), axis=1))

    probs = np.vstack(probs_list)
    gabarito = np.array(gabarito_list)
    m = calcular_metricas_completas(probs, gabarito)
    print(
        f"✅ {desc:<38} | Top-1: {m['acc_top1']:.2f}% | Top-2: {m['acc_top2']:.2f}% | "
        f"F1-Macro: {m['f1_macro']:.2f}% | Kappa: {m['kappa']:.2f}% | ECE: {m['ece']:.2f}%"
    )
    return m

def avaliar_tflite_completo(model_path, val_dataset, desc):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zp = input_details['quantization']
    out_scale, out_zp = output_details['quantization']

    probs_list, gabarito_list = [], []
    t0 = time.perf_counter()

    for images, labels in val_dataset:
        imgs_np = images.numpy()
        lbls_np = labels.numpy()
        for i in range(len(imgs_np)):
            img_int8 = np.round(imgs_np[i] / in_scale + in_zp).clip(-128, 127).astype(np.int8)
            interpreter.set_tensor(input_details['index'], np.expand_dims(img_int8, axis=0))
            interpreter.invoke()
            out_int8 = interpreter.get_tensor(output_details['index'])[0].astype(np.float32)
            prob = np.clip((out_int8 - out_zp) * out_scale, 0.0, 1.0)
            soma = np.sum(prob)
            if soma > 0:
                prob = prob / soma
            probs_list.append(prob)
            gabarito_list.append(np.argmax(lbls_np[i]))

    host_latency_ms = ((time.perf_counter() - t0) / len(gabarito_list)) * 1000.0
    probs = np.vstack(probs_list)
    gabarito = np.array(gabarito_list)
    m = calcular_metricas_completas(probs, gabarito)
    m["host_latency_ms"] = host_latency_ms
    m["quant_params"] = (in_scale, in_zp, out_scale, out_zp)

    print(
        f"✅ {desc:<38} | Top-1: {m['acc_top1']:.2f}% | Top-2: {m['acc_top2']:.2f}% | "
        f"F1-Macro: {m['f1_macro']:.2f}% | Kappa: {m['kappa']:.2f}% | ECE: {m['ece']:.2f}%"
    )
    return m

print("=" * 118)
print(f"AVALIAÇÃO PADRONIZADA MULTI-RESOLUÇÃO ({MODO_EXPERIMENTO_RESOLUCAO}): FLOAT32 vs. TFLITE INT8 (QAT)")
print("=" * 118)

metricas_f32 = {}
metricas_int8 = {}

for alpha in [0.35, 0.50, 0.75]:
    cfg = resultados_treino[alpha]["cfg"]
    res_str = f"{cfg['img_size'][0]}x{cfg['img_size'][1]}"
    _, val_ds_norm_escala = obter_pipelines_por_resolucao(cfg["img_size"])

    metricas_f32[alpha] = avaliar_keras_completo(
        resultados_treino[alpha]["model_f32"], val_ds_norm_escala, f"MobileNetV2 α={alpha:.2f} @{res_str} (Float32)"
    )
    metricas_int8[alpha] = avaliar_tflite_completo(
        auditoria_hw[alpha]["tflite_path"], val_ds_norm_escala, f"MobileNetV2 α={alpha:.2f} @{res_str} (Int8 QAT)"
    )
    params_m = auditoria_hw[alpha]["total_params"] / 1e6
    macs_m = auditoria_hw[alpha]["mmacs"]
    acc_val = metricas_int8[alpha]["acc_top1"]
    auditoria_hw[alpha]["netscore_db"] = 20.0 * math.log10((acc_val ** 2) / ((params_m ** 0.5) * (macs_m ** 0.5)))
    print("-" * 118)

### 5.1 Tabelas Comparativas: Desempenho por Classe, Matriz de Co-Design ESP32-S3 e Validação com o Estado da Arte

In [ ]:
# --- TABELA 1: Comparativo por Classe (Recall % e Degradação de Quantização Δ) ---
linhas_classe = []
for i, cls in enumerate(class_names):
    linhas_classe.append({
        "Classe / Patologia": cls,
        "Amostras Val": contagem_val_limpo[cls],
        f"α=0.35@{RESOLUCOES_ATIVAS[0.35][0]} F32": f"{metricas_f32[0.35]['acc_por_classe'][i]:.1f}%",
        f"α=0.35@{RESOLUCOES_ATIVAS[0.35][0]} Int8": f"{metricas_int8[0.35]['acc_por_classe'][i]:.1f}%",
        "Δ (0.35)": f"{metricas_int8[0.35]['acc_por_classe'][i] - metricas_f32[0.35]['acc_por_classe'][i]:+.1f}%",
        f"α=0.50@{RESOLUCOES_ATIVAS[0.50][0]} F32": f"{metricas_f32[0.50]['acc_por_classe'][i]:.1f}%",
        f"α=0.50@{RESOLUCOES_ATIVAS[0.50][0]} Int8": f"{metricas_int8[0.50]['acc_por_classe'][i]:.1f}%",
        "Δ (0.50)": f"{metricas_int8[0.50]['acc_por_classe'][i] - metricas_f32[0.50]['acc_por_classe'][i]:+.1f}%",
        f"α=0.75@{RESOLUCOES_ATIVAS[0.75][0]} F32": f"{metricas_f32[0.75]['acc_por_classe'][i]:.1f}%",
        f"α=0.75@{RESOLUCOES_ATIVAS[0.75][0]} Int8": f"{metricas_int8[0.75]['acc_por_classe'][i]:.1f}%",
        "Δ (0.75)": f"{metricas_int8[0.75]['acc_por_classe'][i] - metricas_f32[0.75]['acc_por_classe'][i]:+.1f}%",
    })

for titulo_resumo, chave in [
    ("[ACURÁCIA TOP-1 GLOBAL]", "acc_top1"),
    ("[ACURÁCIA TOP-2 GLOBAL]", "acc_top2"),
    ("[F1-SCORE MACRO]", "f1_macro"),
    ("[KAPPA DE COHEN (κ)]", "kappa"),
    ("[MATTHEWS CORR. (MCC)]", "mcc"),
]:
    linhas_classe.append({
        "Classe / Patologia": titulo_resumo,
        "Amostras Val": sum(contagem_val_limpo.values()),
        f"α=0.35@{RESOLUCOES_ATIVAS[0.35][0]} F32": f"{metricas_f32[0.35][chave]:.2f}%",
        f"α=0.35@{RESOLUCOES_ATIVAS[0.35][0]} Int8": f"{metricas_int8[0.35][chave]:.2f}%",
        "Δ (0.35)": f"{metricas_int8[0.35][chave] - metricas_f32[0.35][chave]:+.2f}%",
        f"α=0.50@{RESOLUCOES_ATIVAS[0.50][0]} F32": f"{metricas_f32[0.50][chave]:.2f}%",
        f"α=0.50@{RESOLUCOES_ATIVAS[0.50][0]} Int8": f"{metricas_int8[0.50][chave]:.2f}%",
        "Δ (0.50)": f"{metricas_int8[0.50][chave] - metricas_f32[0.50][chave]:+.2f}%",
        f"α=0.75@{RESOLUCOES_ATIVAS[0.75][0]} F32": f"{metricas_f32[0.75][chave]:.2f}%",
        f"α=0.75@{RESOLUCOES_ATIVAS[0.75][0]} Int8": f"{metricas_int8[0.75][chave]:.2f}%",
        "Δ (0.75)": f"{metricas_int8[0.75][chave] - metricas_f32[0.75][chave]:+.2f}%",
    })

df_classes = pd.DataFrame(linhas_classe)
print("📋 TABELA 1 — Desempenho por Patologia e Resiliência à Quantização (Float32 vs. Int8 QAT):")
display(df_classes)

# --- TABELA 2: Matriz de Co-Design Hardware/Software para o ESP32-S3 ---
linhas_hw = []
for alpha in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[alpha]
    m_i8 = metricas_int8[alpha]
    arena_status = "OK <=800KB" if hw["fits_arena_800kb"] else ("Ajustar <=2MB PSRAM" if hw["fits_arena_2mb"] else "Excede 2MB")
    linhas_hw.append({
        "Modelo (Int8 QAT)": f"MobileNetV2 α={alpha:.2f}",
        "Resolução / Grid": f"{hw['img_size'][0]}x{hw['img_size'][1]} ({hw['grid_final']})",
        "Parâmetros": f"{hw['total_params']/1e3:.1f}k",
        "Complexidade (MMACs)": f"{hw['mmacs']:.1f}M ({hw['mmacs']/ESP32S3_REF_A035_160_MMACS:.2f}x v09)",
        "Flash .tflite (Lim. 1536 KB)": f"{hw['kb_size']:.1f} KB ({hw['flash_usage_pct']:.1f}%)",
        "Tensor Arena PSRAM Est.": f"{hw['estimated_arena_kb']:.0f} KB ({arena_status})",
        "Scratch esp-nn SRAM (48 KB)": f"{hw['esp_nn_scratch_kb']:.1f} KB ({'OK' if hw['fits_sram_48kb'] else 'EXCEDE'})",
        "Latência ESP32-S3 Est.": f"{hw['esp32s3_latency_pure_ms']:.0f} ms ({hw['esp32s3_fps']:.2f} FPS)",
        "Energia / Inf.": f"{hw['energy_mj']:.0f} mJ",
        "Acc Top-1 Int8": f"{m_i8['acc_top1']:.2f}%",
        "Acc Top-2 Int8": f"{m_i8['acc_top2']:.2f}%",
        "F1-Macro Int8": f"{m_i8['f1_macro']:.2f}%",
        "ECE": f"{m_i8['ece']:.2f}%",
        "NetScore (Ω)": f"{hw['netscore_db']:.1f} dB",
    })

df_codesign = pd.DataFrame(linhas_hw)
print("\n⚙️ TABELA 2 — Matriz de Co-Design de Engenharia Embarcada (ESP32-S3 @ 240 MHz + PSRAM Octal 80 MHz):")
display(df_codesign)

# --- TABELA 3: Posicionamento frente ao Estado da Arte e Baseline v08/v09 ---
sota_referencias = [
    {
        "Arquitetura / Referência": "VGG-16 (Militante et al., 2019 / Lit. Cana)",
        "Plataforma Alvo": "GPU Desktop / Cloud",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 528.0,
        "MMACs": 15470.0,
        "Acurácia Reportada (%)": 95.0,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (344x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "ResNet-50 (Daphal & Koli, 2023 / Transfer Learning)",
        "Plataforma Alvo": "GPU / Workstation",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 94.0,
        "MMACs": 3860.0,
        "Acurácia Reportada (%)": 93.8,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (61x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "DenseNet-121 (Benchmark Fitopatologia SOTA)",
        "Plataforma Alvo": "GPU / Edge TPU",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 31.2,
        "MMACs": 2830.0,
        "Acurácia Reportada (%)": 94.2,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (20x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "MobileNetV2 α=1.0 Padrão (Sandler et al. / SOTA Mobile)",
        "Plataforma Alvo": "Smartphone / Raspberry Pi",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 13.6,
        "MMACs": 300.0,
        "Acurácia Reportada (%)": 93.5,
        "Viabilidade MCU ESP32-S3": "❌ Inviável em Float32 (8,8x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "Baseline v08 em Silício (ESP32-S3, α=0.35, 550 limpas)",
        "Plataforma Alvo": "ESP32-S3 N8R16 (Em Silício)",
        "Precisão Numérica": "Int8 QAT (8-bit)",
        "Resolução": "160x160",
        "Tamanho Modelo (MB)": 0.61,
        "MMACs": 30.2,
        "Acurácia Reportada (%)": 91.09,
        "Viabilidade MCU ESP32-S3": "✅ Validado em Hardware (283 ms / 91,09% limpo)",
    },
]

for alpha in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[alpha]
    m_i8 = metricas_int8[alpha]
    lat = hw["esp32s3_latency_pure_ms"]
    status_mcu = (
        f"✅ Tempo Real (<500 ms: ~{lat:.0f} ms)" if (hw["fits_flash"] and lat <= 500.0) else
        (f"✅ Voo Pairado (~{lat:.0f} ms)" if (hw["fits_flash"] and lat <= 700.0) else f"⚠️ Bancada / Offline (~{lat:.0f} ms)")
    )
    sota_referencias.append({
        "Arquitetura / Referência": f"★ Este Trabalho (v10): MobileNetV2 α={alpha:.2f} (Int8 QAT)",
        "Plataforma Alvo": "ESP32-S3 N8R16 (TFLite Micro + esp-nn)",
        "Precisão Numérica": "Int8 QAT (8-bit)",
        "Resolução": f"{hw['img_size'][0]}x{hw['img_size'][1]}",
        "Tamanho Modelo (MB)": round(hw["kb_size"] / 1024.0, 2),
        "MMACs": round(hw["mmacs"], 1),
        "Acurácia Reportada (%)": round(m_i8["acc_top1"], 2),
        "Viabilidade MCU ESP32-S3": status_mcu,
    })

df_sota = pd.DataFrame(sota_referencias)
print("\n🏆 TABELA 3 — Validação contra o Estado da Arte (Literatura vs. Baseline v08 vs. Modelos v10):")
display(df_sota)

---
## 6. Visualizações Avançadas de Desempenho, Controle de Overfitting, Fronteira de Pareto e Estado da Arte

In [ ]:
# --- GRÁFICO 1: Curvas de Aprendizado Unificadas (Fase 1 + Fase 2A + Fase 2B Clean Cooldown + Fase 3 QAT) ---
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11.5, 'axes.labelsize': 10.5})
fig, axes = plt.subplots(3, 2, figsize=(19, 15))

cores_escala = {
    0.35: ('#1b5e20', '#4caf50'),
    0.50: ('#0d47a1', '#2196f3'),
    0.75: ('#4a148c', '#9c27b0'),
}

for row_idx, alpha in enumerate([0.35, 0.50, 0.75]):
    res = resultados_treino[alpha]
    cfg = res["cfg"]
    cor_tr, cor_lig = cores_escala[alpha]

    h1 = res["hist_p1"].history
    h2a = res["hist_p2a"].history
    h2b = res["hist_p2b"].history
    h3 = res["hist_qat"].history

    acc_tr = h1['accuracy'] + h2a['accuracy'] + h2b['accuracy'] + h3['accuracy']
    acc_val = h1['val_accuracy'] + h2a['val_accuracy'] + h2b['val_accuracy'] + h3['val_accuracy']
    loss_tr = h1['loss'] + h2a['loss'] + h2b['loss'] + h3['loss']
    loss_val = h1['val_loss'] + h2a['val_loss'] + h2b['val_loss'] + h3['val_loss']

    ep_p2a = len(h1['accuracy'])
    ep_p2b = ep_p2a + len(h2a['accuracy'])
    ep_qat = ep_p2b + len(h2b['accuracy'])
    epocas = np.arange(len(acc_tr))

    gap_final = (acc_tr[-1] - acc_val[-1]) * 100.0

    # Subplot Acurácia
    ax_acc = axes[row_idx, 0]
    ax_acc.plot(epocas, acc_tr, label='Treino', color=cor_tr, linewidth=2.0)
    ax_acc.plot(epocas, acc_val, label='Validação', color='#ef6c00', linewidth=2.2)
    ax_acc.fill_between(epocas, acc_tr, acc_val, color=cor_lig, alpha=0.12, label=f'Gap Final: {gap_final:+.1f}%')
    ax_acc.axvline(ep_p2a - 0.5, color='#6a1b9a', linestyle='-.', linewidth=1.2, label='Fase 2A (Fine-Tune)')
    ax_acc.axvline(ep_p2b - 0.5, color='#00838f', linestyle=':', linewidth=1.6, label='Fase 2B (Clean Cooldown)')
    ax_acc.axvline(ep_qat - 0.5, color='#c62828', linestyle='--', linewidth=1.5, label='Fase 3 (QAT Int8)')
    ax_acc.set_title(
        f"{cfg['titulo']} — Acurácia (SpatialDrop={cfg['spatial_dropout_rate']}, Drop={cfg['dropout_rate']})",
        fontweight='bold'
    )
    ax_acc.set_xlabel('Época Acumulada')
    ax_acc.set_ylabel('Acurácia')
    ax_acc.set_ylim(0.25, 1.03)
    ax_acc.grid(True, linestyle=':', alpha=0.6)
    ax_acc.legend(loc='lower right', fontsize=8.0)

    # Subplot Loss
    ax_loss = axes[row_idx, 1]
    ax_loss.plot(epocas, loss_tr, label='Treino', color=cor_tr, linewidth=2.0)
    ax_loss.plot(epocas, loss_val, label='Validação', color='#ef6c00', linewidth=2.2)
    ax_loss.axvline(ep_p2a - 0.5, color='#6a1b9a', linestyle='-.', linewidth=1.2, label='Fase 2A (Fine-Tune)')
    ax_loss.axvline(ep_p2b - 0.5, color='#00838f', linestyle=':', linewidth=1.6, label='Fase 2B (Clean Cooldown)')
    ax_loss.axvline(ep_qat - 0.5, color='#c62828', linestyle='--', linewidth=1.5, label='Fase 3 (QAT Int8)')
    ax_loss.set_title(
        f"{cfg['titulo']} — Focal Loss (L2={cfg['l2_weight_decay']}, Smooth={cfg['label_smoothing']})",
        fontweight='bold'
    )
    ax_loss.set_xlabel('Época Acumulada')
    ax_loss.set_ylabel('Focal Crossentropy Loss')
    ax_loss.grid(True, linestyle=':', alpha=0.6)
    ax_loss.legend(loc='upper right', fontsize=8.0)

plt.suptitle(
    f'Gráfico 1 — Dinâmica de Convergência em 4 Estágios (v10: {MODO_EXPERIMENTO_RESOLUCAO})',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 2: Matrizes de Confusão Normalizadas (Grid 2x3: Float32 vs. Int8 QAT) ---
fig, axes = plt.subplots(2, 3, figsize=(24, 14.5))

for col_idx, alpha in enumerate([0.35, 0.50, 0.75]):
    m_f32 = metricas_f32[alpha]
    m_i8 = metricas_int8[alpha]
    hw = auditoria_hw[alpha]
    kb = hw["kb_size"]
    res_lbl = f"{hw['img_size'][0]}x{hw['img_size'][1]}"
    delta_acc = m_i8["acc_top1"] - m_f32["acc_top1"]
    cmap = "Greens" if alpha == 0.35 else ("Blues" if alpha == 0.50 else "Purples")

    pares = [
        (
            m_f32["cm"],
            f"MobileNetV2 α={alpha:.2f} @{res_lbl} (Float32)\nTop-1: {m_f32['acc_top1']:.2f}% | F1-Macro: {m_f32['f1_macro']:.2f}%",
            axes[0, col_idx],
            cmap
        ),
        (
            m_i8["cm"],
            f"MobileNetV2 α={alpha:.2f} @{res_lbl} (Int8 QAT — {kb:.0f} KB)\nTop-1: {m_i8['acc_top1']:.2f}% | F1: {m_i8['f1_macro']:.2f}% (Δ: {delta_acc:+.2f}%)",
            axes[1, col_idx],
            cmap
        ),
    ]

    for cm, titulo, ax, mapa_cor in pares:
        cm_perc = (cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], 1)) * 100.0
        annot_matrix = np.empty(cm.shape, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                val_p = cm_perc[i, j]
                val_c = cm[i, j]
                annot_matrix[i, j] = f"{val_p:.0f}%\n({val_c})" if val_c > 0 else "0"

        sns.heatmap(
            cm_perc, annot=annot_matrix, fmt='', cmap=mapa_cor,
            vmin=0, vmax=100, cbar=True, ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Recall (%)'},
            annot_kws={'size': 7.5}
        )
        ax.set_title(titulo, fontweight='bold', pad=10, fontsize=11)
        ax.set_xlabel('Classe Predita', fontweight='bold')
        ax.set_ylabel('Classe Real', fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8.5)
        ax.tick_params(axis='y', rotation=0, labelsize=8.5)

plt.suptitle(
    'Gráfico 2 — Matrizes de Confusão Normalizadas (% de Recall + Contagem Absoluta): Float32 vs. Int8 QAT (v10)',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 3: Comparativo Direto por Classe e Métricas Globais (Modelos Int8 QAT vs. Meta SOTA 90%) ---
fig, ax = plt.subplots(figsize=(18, 7.5))

labels_eixo = class_names + ['[TOP-1 GLOBAL]', '[TOP-2 GLOBAL]', '[F1-MACRO]', '[KAPPA κ]']

def vetor_barras_int8(alpha):
    m = metricas_int8[alpha]
    return np.concatenate([m['acc_por_classe'], [m['acc_top1'], m['acc_top2'], m['f1_macro'], m['kappa']]])

v_035 = vetor_barras_int8(0.35)
v_050 = vetor_barras_int8(0.50)
v_075 = vetor_barras_int8(0.75)

x = np.arange(len(labels_eixo))
largura = 0.26

b1 = ax.bar(
    x - largura, v_035, largura,
    label=f"MobileNetV2 α=0.35 @{auditoria_hw[0.35]['img_size'][0]} Int8 ({auditoria_hw[0.35]['kb_size']:.0f} KB | ~{auditoria_hw[0.35]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#2e7d32', edgecolor='black', linewidth=0.6
)
b2 = ax.bar(
    x, v_050, largura,
    label=f"MobileNetV2 α=0.50 @{auditoria_hw[0.50]['img_size'][0]} Int8 ({auditoria_hw[0.50]['kb_size']:.0f} KB | ~{auditoria_hw[0.50]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#1565c0', edgecolor='black', linewidth=0.6
)
b3 = ax.bar(
    x + largura, v_075, largura,
    label=f"MobileNetV2 α=0.75 @{auditoria_hw[0.75]['img_size'][0]} Int8 ({auditoria_hw[0.75]['kb_size']:.0f} KB | ~{auditoria_hw[0.75]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#6a1b9a', edgecolor='black', linewidth=0.6
)

ax.axhline(90.0, color='#d32f2f', linestyle='--', linewidth=1.5, alpha=0.85, label='Limiar SOTA de Produção Agrícola (90%)')
ax.axvline(len(class_names) - 0.5, color='black', linestyle=':', linewidth=1.2, alpha=0.6)

ax.set_ylabel('Desempenho Int8 (%)', fontweight='bold')
ax.set_title(
    'Gráfico 3 — Desempenho por Patologia e Indicadores Globais dos Modelos v10 Quantizados Int8',
    fontsize=13.5, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels(labels_eixo, rotation=28, ha='right', fontweight='bold')
ax.set_ylim(0, 115)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.95, fontsize=9.5)

for bar_group in [b1, b2, b3]:
    for rect in bar_group:
        h = rect.get_height()
        ax.annotate(
            f'{h:.1f}%',
            xy=(rect.get_x() + rect.get_width() / 2.0, h),
            xytext=(0, 3), textcoords="offset points",
            ha='center', va='bottom', fontsize=7.0, fontweight='bold', rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 4: Dashboard de Engenharia Embarcada ESP32-S3 (Pareto, Memória, Rendimentos e Calibração) ---
fig, axes = plt.subplots(2, 2, figsize=(19, 13.5))

alphas = [0.35, 0.50, 0.75]
cores_barras = ['#2e7d32', '#1565c0', '#6a1b9a']

# 4A: Fronteira de Pareto
ax_p = axes[0, 0]
lats = [auditoria_hw[a]["esp32s3_latency_pure_ms"] for a in alphas]
accs_i8 = [metricas_int8[a]["acc_top1"] for a in alphas]
kbs = [auditoria_hw[a]["kb_size"] for a in alphas]
max_x_pareto = max(1400.0, max(lats) * 1.25)

ax_p.axvspan(0, 420, color='#c8e6c9', alpha=0.35, label='Zona Tempo Real (< 420 ms | >= 2.4 FPS)')
ax_p.axvspan(420, 700, color='#fff9c4', alpha=0.45, label='Zona Voo Pairado (420–700 ms)')
ax_p.axvspan(700, max_x_pareto, color='#ffcdd2', alpha=0.35, label='Zona Bancada / Alta Latência (> 700 ms)')
ax_p.axhline(90.0, color='#d32f2f', linestyle='--', linewidth=1.2, label='Meta Acurácia SOTA (90%)')

ax_p.plot(lats, accs_i8, color='#37474f', linestyle='-.', linewidth=1.8, zorder=3)
for idx, a in enumerate(alphas):
    res_lbl = f"{auditoria_hw[a]['img_size'][0]}x{auditoria_hw[a]['img_size'][1]}"
    ax_p.scatter(
        lats[idx], accs_i8[idx],
        s=kbs[idx] * 0.45, color=cores_barras[idx], edgecolor='black', linewidth=1.5, alpha=0.9, zorder=4
    )
    ax_p.annotate(
        f"v10 α={a:.2f} @{res_lbl}\nAcc: {accs_i8[idx]:.2f}% | {kbs[idx]:.0f} KB\n~{lats[idx]:.0f} ms ({auditoria_hw[a]['energy_mj']:.0f} mJ)",
        xy=(lats[idx], accs_i8[idx]),
        xytext=(14 if idx < 2 else -145, -18 if idx == 0 else 14),
        textcoords="offset points",
        fontsize=9, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=cores_barras[idx], alpha=0.9),
        arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0.15", color=cores_barras[idx])
    )

ax_p.set_title('4A — Fronteira de Pareto no ESP32-S3: Latência vs. Acurácia Int8 (v10)', fontweight='bold')
ax_p.set_xlabel('Latência de Inferência Estimada no ESP32-S3 @ 240 MHz (ms)', fontweight='bold')
ax_p.set_ylabel('Acurácia Global Top-1 Int8 (%)', fontweight='bold')
ax_p.set_xlim(150, max_x_pareto)
ax_p.set_ylim(min(accs_i8) - 4.0, min(100.0, max(accs_i8) + 3.5))
ax_p.grid(True, linestyle=':', alpha=0.6)
ax_p.legend(loc='lower right', fontsize=8.5)

# 4B: Hierarquia de Memória (SRAM 48 KB, Arena 800 KB, Flash 1536 KB)
ax_m = axes[0, 1]
x_m = np.arange(len(alphas))
w_m = 0.25

pct_sram = [(auditoria_hw[a]["esp_nn_scratch_kb"] / ESP32S3_SRAM_SCRATCH_LIMIT_KB) * 100.0 for a in alphas]
pct_arena = [(auditoria_hw[a]["estimated_arena_kb"] / ESP32S3_ARENA_BUDGET_KB) * 100.0 for a in alphas]
pct_flash = [auditoria_hw[a]["flash_usage_pct"] for a in alphas]

ax_m.bar(x_m - w_m, pct_sram, w_m, label='Scratch SIMD esp-nn (Limite 48 KB SRAM Interna)', color='#00897b', edgecolor='black')
ax_m.bar(x_m, pct_arena, w_m, label='Tensor Arena Ativações (Ref. 800 KB PSRAM Padrão)', color='#fb8c00', edgecolor='black')
ax_m.bar(x_m + w_m, pct_flash, w_m, label='Modelo .tflite (Partição 1.536 KB Flash LittleFS)', color='#3949ab', edgecolor='black')

ax_m.axhline(100.0, color='#d32f2f', linestyle='--', linewidth=1.8, label='Teto 100% (SRAM 48KB / Arena Padrão 800KB / Flash 1.5MB)')
ax_m.set_title('4B — Ocupação da Hierarquia de Memória do ESP32-S3 (% do Orçamento)', fontweight='bold')
ax_m.set_xticks(x_m)
ax_m.set_xticklabels([f"α={a:.2f} @{auditoria_hw[a]['img_size'][0]}" for a in alphas], fontweight='bold')
ax_m.set_ylabel('Ocupação do Orçamento (%)', fontweight='bold')
ax_m.set_ylim(0, max(max(pct_flash), max(pct_arena), 115) * 1.18)
ax_m.grid(axis='y', linestyle=':', alpha=0.6)
ax_m.legend(loc='upper left', fontsize=8.3)

for idx, a in enumerate(alphas):
    ax_m.annotate(f"{auditoria_hw[a]['esp_nn_scratch_kb']:.1f}KB\n({pct_sram[idx]:.0f}%)",
                  xy=(x_m[idx] - w_m, pct_sram[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax_m.annotate(f"{auditoria_hw[a]['estimated_arena_kb']:.0f}KB\n({pct_arena[idx]:.0f}%)",
                  xy=(x_m[idx], pct_arena[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax_m.annotate(f"{auditoria_hw[a]['kb_size']:.0f}KB\n({pct_flash[idx]:.0f}%)",
                  xy=(x_m[idx] + w_m, pct_flash[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')

# 4C: Fidelidade de Quantização e NetScore
ax_q = axes[1, 0]
x_q = np.arange(len(alphas))
w_q = 0.26

accs_f32 = [metricas_f32[a]["acc_top1"] for a in alphas]
accs_top2 = [metricas_int8[a]["acc_top2"] for a in alphas]

ax_q.bar(x_q - w_q, accs_f32, w_q, label='Acurácia Top-1 Float32 (Pós Fase 2B)', color='#90caf9', edgecolor='black')
ax_q.bar(x_q, accs_i8, w_q, label='Acurácia Top-1 Int8 (Pós QAT)', color='#1e88e5', edgecolor='black')
ax_q.bar(x_q + w_q, accs_top2, w_q, label='Acurácia Top-2 Int8 (Apoio à Decisão)', color='#43a047', edgecolor='black')

ax_q2 = ax_q.twinx()
netscores = [auditoria_hw[a]["netscore_db"] for a in alphas]
ax_q2.plot(x_q, netscores, color='#d81b60', marker='D', markersize=8, linewidth=2.2, label='NetScore Ω (dB)')
ax_q2.set_ylabel('NetScore Ω (dB — Maior é Mais Eficiente)', color='#d81b60', fontweight='bold')
ax_q2.tick_params(axis='y', labelcolor='#d81b60')
ax_q2.set_ylim(min(netscores) - 5, max(netscores) + 5)

for idx, a in enumerate(alphas):
    delta = accs_i8[idx] - accs_f32[idx]
    ax_q.annotate(f"{accs_i8[idx]:.2f}%\n(Δ:{delta:+.2f}%)",
                  xy=(x_q[idx], accs_i8[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_q.annotate(f"{accs_top2[idx]:.1f}%",
                  xy=(x_q[idx] + w_q, accs_top2[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8.0, fontweight='bold')

ax_q.set_title('4C — Fidelidade de Quantização (Float32 vs. Int8), Top-2 e NetScore (Ω)', fontweight='bold')
ax_q.set_xticks(x_q)
ax_q.set_xticklabels([f"α={a:.2f} @{auditoria_hw[a]['img_size'][0]}\n({auditoria_hw[a]['mmacs']:.0f} MMACs)" for a in alphas], fontweight='bold')
ax_q.set_ylabel('Acurácia (%)', fontweight='bold')
ax_q.set_ylim(75, 105)
ax_q.grid(axis='y', linestyle=':', alpha=0.6)

lines_1, labels_1 = ax_q.get_legend_handles_labels()
lines_2, labels_2 = ax_q2.get_legend_handles_labels()
ax_q.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower right', fontsize=8.5)

# 4D: Calibração Probabilística Int8
ax_c = axes[1, 1]
conf_ok = [metricas_int8[a]["conf_corretos"] for a in alphas]
conf_err = [metricas_int8[a]["conf_errados"] for a in alphas]
eces = [metricas_int8[a]["ece"] for a in alphas]

ax_c.bar(x_q - w_q, conf_ok, w_q, label='Confiança Média nos Acertos (%)', color='#2e7d32', edgecolor='black')
ax_c.bar(x_q, conf_err, w_q, label='Confiança Média nos Erros (%) [Menor é Melhor]', color='#ef6c00', edgecolor='black')
ax_c.bar(x_q + w_q, eces, w_q, label='Expected Calibration Error — ECE (%) [Menor é Melhor]', color='#c62828', edgecolor='black')

for idx in range(len(alphas)):
    ax_c.annotate(f"{conf_ok[idx]:.1f}%", xy=(x_q[idx] - w_q, conf_ok[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_c.annotate(f"{conf_err[idx]:.1f}%", xy=(x_q[idx], conf_err[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_c.annotate(f"ECE: {eces[idx]:.2f}%", xy=(x_q[idx] + w_q, eces[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax_c.set_title('4D — Calibração Probabilística Pós-Quantização Int8 (Acertos vs. Erros e ECE)', fontweight='bold')
ax_c.set_xticks(x_q)
ax_c.set_xticklabels([f"α={a:.2f} @{auditoria_hw[a]['img_size'][0]}\n(Smooth={PERFIS_TREINAMENTO[a]['label_smoothing']})" for a in alphas], fontweight='bold')
ax_c.set_ylabel('Probabilidade / Erro de Calibração (%)', fontweight='bold')
ax_c.set_ylim(0, 110)
ax_c.grid(axis='y', linestyle=':', alpha=0.6)
ax_c.legend(loc='upper right', fontsize=8.5)

plt.suptitle(
    'Gráfico 4 — Co-Design Hardware/Software para o ESP32-S3 (v10 Multi-Resolução)',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 5 & 6: Posicionamento frente ao Estado da Arte (Literatura) e Radar Multidimensional ---
fig = plt.figure(figsize=(20, 8.5))

ax_sota = fig.add_subplot(1, 2, 1)
alphas = [0.35, 0.50, 0.75]
cores_barras = ['#2e7d32', '#1565c0', '#6a1b9a']
ax_sota.axvspan(0.1, 1.5, color='#e8f5e9', alpha=0.55, label='Zona Viável Flash ESP32-S3 (<= 1,5 MB LittleFS)')
ax_sota.axvline(1.5, color='#2e7d32', linestyle='--', linewidth=1.8)
ax_sota.axhline(90.0, color='#d32f2f', linestyle=':', linewidth=1.5, label='Referência SOTA Produção (90% Acc)')

pontos_literatura = [
    ("VGG-16 (Lit. Float32)", 528.0, 95.0, '#78909c', 's', 130),
    ("ResNet-50 (Daphal & Koli, F32)", 94.0, 93.8, '#78909c', '^', 130),
    ("DenseNet-121 (Lit. Float32)", 31.2, 94.2, '#78909c', 'v', 130),
    ("MobileNetV2 α=1.0 (Float32)", 13.6, 93.5, '#546e7a', 'o', 140),
    ("Baseline v08 ESP32-S3 (160x160)", 0.61, 91.09, '#00897b', 'P', 180),
]

for nome_p, mb_p, acc_p, cor_p, mk_p, sz_p in pontos_literatura:
    ax_sota.scatter(mb_p, acc_p, color=cor_p, marker=mk_p, s=sz_p, edgecolor='black', linewidth=1.2, zorder=4)
    offset_y = -15 if "Baseline" in nome_p or "ResNet" in nome_p else 8
    ax_sota.annotate(
        f"{nome_p}\n({mb_p:.2f} MB | {acc_p:.1f}%)",
        xy=(mb_p, acc_p), xytext=(0, offset_y), textcoords="offset points",
        ha='center', fontsize=8.0, fontweight='bold'
    )

for idx, a in enumerate(alphas):
    mb_val = auditoria_hw[a]["kb_size"] / 1024.0
    acc_val = metricas_int8[a]["acc_top1"]
    res_lbl = f"{auditoria_hw[a]['img_size'][0]}px"
    ax_sota.scatter(
        mb_val, acc_val, color=cores_barras[idx], marker='*', s=300,
        edgecolor='black', linewidth=1.3, zorder=5,
        label=f"v10 α={a:.2f} @{res_lbl} Int8 ({mb_val:.2f} MB | {acc_val:.2f}%)"
    )
    ax_sota.annotate(
        f"★ v10 α={a:.2f} @{res_lbl}\n({mb_val:.2f} MB | {acc_val:.1f}%)",
        xy=(mb_val, acc_val), xytext=(-15 if idx == 0 else 15, 10 if idx != 1 else -18),
        textcoords="offset points", ha='center', fontsize=8.8, fontweight='bold',
        color=cores_barras[idx]
    )

ax_sota.set_xscale('log')
ax_sota.set_xlim(0.25, 1200.0)
ax_sota.set_ylim(84.0, 98.0)
ax_sota.set_title('Gráfico 5 — Posicionamento frente ao Estado da Arte (Tamanho Log MB vs. Acurácia %)', fontweight='bold')
ax_sota.set_xlabel('Tamanho do Modelo em Memória (MB — Escala Logarítmica)', fontweight='bold')
ax_sota.set_ylabel('Acurácia Global Top-1 (%)', fontweight='bold')
ax_sota.grid(True, which='both', linestyle=':', alpha=0.6)
ax_sota.legend(loc='lower right', fontsize=8.2)

ax_rad = fig.add_subplot(1, 2, 2, polar=True)
categorias_radar = [
    'Acurácia Top-1\n(Int8)',
    'F1-Macro\n(Equilíbrio)',
    'Velocidade FPS\n(ESP32-S3)',
    'Economia de\nFlash (1.5MB)',
    'Margem SRAM\n(esp-nn 48KB)',
    'Eficiência\nNetScore (Ω)'
]
N_eixos = len(categorias_radar)
angulos = np.linspace(0, 2 * np.pi, N_eixos, endpoint=False).tolist()
angulos += angulos[:1]

for idx, a in enumerate(alphas):
    hw = auditoria_hw[a]
    m = metricas_int8[a]
    score_acc = m["acc_top1"]
    score_f1 = m["f1_macro"]
    score_fps = min(100.0, (auditoria_hw[0.35]["esp32s3_latency_pure_ms"] / hw["esp32s3_latency_pure_ms"]) * 95.0)
    score_flash = max(0.0, (1.0 - (hw["kb_size"] / (ESP32S3_FLASH_BUDGET_KB * 1.5))) * 100.0)
    score_sram = max(0.0, (1.0 - (hw["esp_nn_scratch_kb"] / ESP32S3_SRAM_SCRATCH_LIMIT_KB)) * 100.0)
    score_net = min(100.0, (hw["netscore_db"] / max(auditoria_hw[x]["netscore_db"] for x in alphas)) * 95.0)

    valores = [score_acc, score_f1, score_fps, score_flash, score_sram, score_net]
    valores += valores[:1]

    ax_rad.plot(angulos, valores, color=cores_barras[idx], linewidth=2.3, label=f"v10 α={a:.2f} @{hw['img_size'][0]}px")
    ax_rad.fill(angulos, valores, color=cores_barras[idx], alpha=0.14)

ax_rad.set_xticks(angulos[:-1])
ax_rad.set_xticklabels(categorias_radar, fontweight='bold', fontsize=9.5)
ax_rad.set_ylim(0, 105)
ax_rad.set_title('Gráfico 6 — Radar Multidimensional de Co-Design Embarcado (v10)', fontweight='bold', pad=22)
ax_rad.legend(loc='upper right', bbox_to_anchor=(1.25, 1.12), fontsize=9.5)

plt.tight_layout()
plt.show()

---
## 7. Seleção por Índice de Co-Design Pareto e Exportação Automatizada C/C++ (`.h` com `alignas(16)`)

Selecionamos objetivamente a variante padrão para o cabeçalho `model_cane_disease.h`, gravando as dimensões exatas de entrada (`MODEL_INPUT_WIDTH`, `MODEL_INPUT_HEIGHT`) e a recomendação de `kTensorArenaSize` para o firmware C++ do ESP32-S3.

In [ ]:
# Seleciona o modelo recomendado equilibrando Acurácia Int8 e restrições físicas do ESP32-S3
acc_base_035 = metricas_int8[0.35]["acc_top1"]
alpha_escolhido = 0.35

if (
    auditoria_hw[0.50]["kb_size"] <= 1400.0
    and auditoria_hw[0.50]["fits_sram_48kb"]
    and auditoria_hw[0.50]["fits_arena_2mb"]
    and auditoria_hw[0.50]["esp32s3_latency_pure_ms"] <= 680.0
    and (metricas_int8[0.50]["acc_top1"] - acc_base_035) >= 1.5
):
    alpha_escolhido = 0.50

if (
    auditoria_hw[0.75]["kb_size"] <= 1400.0
    and auditoria_hw[0.75]["fits_sram_48kb"]
    and auditoria_hw[0.75]["fits_arena_2mb"]
    and auditoria_hw[0.75]["esp32s3_latency_pure_ms"] <= 680.0
    and (metricas_int8[0.75]["acc_top1"] - metricas_int8[alpha_escolhido]["acc_top1"]) >= 1.5
):
    alpha_escolhido = 0.75

hw_escolhido = auditoria_hw[alpha_escolhido]
cfg_escolhido = PERFIS_TREINAMENTO[alpha_escolhido]
MODEL_ESCOLHIDO = hw_escolhido["tflite_path"]
NOME_SAIDA = "model_cane_disease"
ARENA_RECOMENDADA_KB = max(800, int(math.ceil(hw_escolhido["estimated_arena_kb"] / 64.0) * 64))

print("=" * 100)
print(f"🏆 MODELO SELECIONADO PARA HEADER C/C++ PADRÃO: MobileNetV2 α={alpha_escolhido:.2f} @{cfg_escolhido['img_size'][0]}x{cfg_escolhido['img_size'][1]} ({MODEL_ESCOLHIDO})")
print("   Justificativa de Co-Design:")
for a in [0.35, 0.50, 0.75]:
    marcador = "👉 [SELECIONADO]" if a == alpha_escolhido else "   [ALTERNATIVO]"
    print(
        f"   {marcador} α={a:.2f} @{auditoria_hw[a]['img_size'][0]}: Top-1={metricas_int8[a]['acc_top1']:.2f}% | F1={metricas_int8[a]['f1_macro']:.2f}% | "
        f"Flash={auditoria_hw[a]['kb_size']:.1f} KB | Arena={auditoria_hw[a]['estimated_arena_kb']:.0f} KB | "
        f"Latência=~{auditoria_hw[a]['esp32s3_latency_pure_ms']:.0f} ms | NetScore={auditoria_hw[a]['netscore_db']:.1f} dB"
    )
print("=" * 100)

interp = tf.lite.Interpreter(model_path=MODEL_ESCOLHIDO)
interp.allocate_tensors()
in_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

in_scale, in_zp = in_det['quantization']
out_scale, out_zp = out_det['quantization']

with open(MODEL_ESCOLHIDO, "rb") as f_bin:
    dados_tflite = f_bin.read()

linhas_hex = []
for i in range(0, len(dados_tflite), 12):
    bloco = ", ".join(f"0x{b:02x}" for b in dados_tflite[i:i+12])
    linhas_hex.append(f"  {bloco}")
array_c_bytes = ",\n".join(linhas_hex)
nome_var_c = MODEL_ESCOLHIDO.replace(".", "_")

conteudo_hex = (
    f"unsigned char {nome_var_c}[] = {{\n{array_c_bytes}\n}};\n"
    f"unsigned int {nome_var_c}_len = {len(dados_tflite)};"
)

classes_c_array = ",\n    ".join([f'"{c}"' for c in class_names])

header_completo = f"""#ifndef MODEL_CANE_DISEASE_H
#define MODEL_CANE_DISEASE_H

#include <stdint.h>

// ============================================================================
// Modelo TFLite Int8 Gerado Automaticamente para ESP32-S3 (TFLite Micro + esp-nn)
// Estudo Multi-Resolução v10 — Variante Selecionada: MobileNetV2 alpha={alpha_escolhido:.2f}
// Resolução: {cfg_escolhido['img_size'][0]}x{cfg_escolhido['img_size'][1]} | Arquivo fonte: {MODEL_ESCOLHIDO}
// ============================================================================

#define MODEL_INPUT_WIDTH         {cfg_escolhido['img_size'][1]}
#define MODEL_INPUT_HEIGHT        {cfg_escolhido['img_size'][0]}
#define MODEL_INPUT_CHANNELS      3
#define MODEL_NUM_CLASSES         {NUM_CLASSES}
#define MODEL_RECOMMENDED_ARENA_KB {ARENA_RECOMENDADA_KB}

#define MODEL_INPUT_SCALE         {in_scale:.8f}f
#define MODEL_INPUT_ZERO_POINT    {in_zp}
#define MODEL_OUTPUT_SCALE        {out_scale:.8f}f
#define MODEL_OUTPUT_ZERO_POINT   {out_zp}

static const char* kCategoryLabels[MODEL_NUM_CLASSES] = {{
    {classes_c_array}
}};

// Conversão direta de pixel RGB888 (uint8_t [0..255]) para Int8 [-128..127] sem float:
static inline int8_t quantize_pixel_u8_to_i8(uint8_t pixel_u8) {{
    return (int8_t)((int32_t)pixel_u8 - 128);
}}

alignas(16) {conteudo_hex}

#endif // MODEL_CANE_DISEASE_H
"""

with open(f"{NOME_SAIDA}.h", "w") as f_out:
    f_out.write(header_completo)

print(f"\n✅ Arquivo {NOME_SAIDA}.h gerado com sucesso a partir de '{MODEL_ESCOLHIDO}'!")
print(f"   - Resolução de Entrada: {cfg_escolhido['img_size'][1]}x{cfg_escolhido['img_size'][0]}x3")
print(f"   - Tamanho do modelo   : {len(dados_tflite)/1024:.2f} KB")
print(f"   - Arena Recomendada   : {ARENA_RECOMENDADA_KB} KB (PSRAM)")
print(f"   - Input Quantization  : scale = {in_scale:.8f}, zero_point = {in_zp}")
print(f"   - Output Quantization : scale = {out_scale:.8f}, zero_point = {out_zp}")

---
## 8. Geração Automatizada do Relatório de Treinamento (`training_reports_v10.md`)

Consolida em Markdown todas as métricas experimentais, configurações de resolução por recorte, histórico das 4 fases, desempenho por patologia, matriz de co-design ESP32-S3 e decisão Pareto para documentação e rastreabilidade direta no repositório.

In [ ]:
REPORT_MD_PATH = "training_reports_v10.md"
timestamp_relatorio = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
gpu_disponivel = len(tf.config.list_physical_devices('GPU')) > 0

linhas_md = [
    f"# Relatório Consolidado de Treinamento — Notebook v10 (`training_reports_v10.md`)",
    "",
    f"**Data de Geração:** {timestamp_relatorio}  ",
    f"**Framework:** TensorFlow `{tf.__version__}` (Legacy Keras 2 + `tfmot`) | **GPU Ativa:** `{gpu_disponivel}` | **Seed:** `{SEED}`  ",
    f"**Modo de Experimento:** `{MODO_EXPERIMENTO_RESOLUCAO}` | **Batch Size:** `{BATCH_SIZE}` | **Classes:** `{NUM_CLASSES}`  ",
    f"**Dataset de Origem:** `{CAMINHO_DATASET_ORIGEM}`",
    "",
    "---",
    "",
    "## 1. Resumo da Auditoria MD5 e Split Estratificado (80/20)",
    "",
    f"- **Arquivos Brutos Auditados:** `{auditoria_dataset_md5['total_brutos']}`",
    f"- **Hashes MD5 Únicos:** `{auditoria_dataset_md5['hashes_unicos']}`",
    f"- **Pares com Colisão Cruzada Removidos:** `{auditoria_dataset_md5['pares_colisao']}`",
    f"- **Amostras Limpas Totais:** `{sum(contagem_treino_limpo.values()) + sum(contagem_val_limpo.values())}` (`{sum(contagem_treino_limpo.values())}` treino / `{sum(contagem_val_limpo.values())}` validação)",
    "",
    "---",
    "",
    "## 2. Estratégia Multi-Resolução, Regularização e Tempo de Treinamento (4 Estágios)",
    "",
    "| Recorte ($\\alpha$) | Resolução / Grid | SpatialDrop / Drop / L2 | Freeze (F2A $\\rightarrow$ F2B) | Fases (F1 / F2A / F2B / QAT) | Tempo Total |",
    "| :---: | :---: | :---: | :---: | :---: | :---: |",
]

for a in [0.35, 0.50, 0.75]:
    res = resultados_treino[a]
    c = res["cfg"]
    tfases = res["tempo_fases"]
    ep_str = (
        f"{len(res['hist_p1'].history['accuracy'])} / "
        f"{len(res['hist_p2a'].history['accuracy'])} / "
        f"{len(res['hist_p2b'].history['accuracy'])} / "
        f"{len(res['hist_qat'].history['accuracy'])} ép."
    )
    linhas_md.append(
        f"| `{a:.2f}` | `{c['img_size'][0]}x{c['img_size'][1]}` (`{c['grid_final']}`) | "
        f"`{c['spatial_dropout_rate']}` / `{c['dropout_rate']}` / `{c['l2_weight_decay']}` | "
        f"`{int(c['freeze_ratio_phase2a']*100)}%` $\\rightarrow$ `{int(c['freeze_ratio_phase2b']*100)}%` | "
        f"{ep_str} ({tfases['fase1_min']:.1f}+{tfases['fase2a_min']:.1f}+{tfases['fase2b_min']:.1f}+{tfases['fase3_qat_min']:.1f}m) | "
        f"**{res['tempo_treino_min']:.1f} min** |"
    )

linhas_md.extend([
    "",
    "---",
    "",
    "## 3. Resultados Globais: Float32 vs. TFLite Int8 (QAT)",
    "",
    "| Recorte | Top-1 F32 | Top-1 Int8 | $\\Delta$ (Int8 - F32) | Top-2 Int8 | F1-Macro Int8 | Precisão / Recall Macro | Kappa ($\\kappa$) | MCC | Conf. Acertos / Erros | ECE Int8 |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |",
])

for a in [0.35, 0.50, 0.75]:
    m32 = metricas_f32[a]
    mi8 = metricas_int8[a]
    c = PERFIS_TREINAMENTO[a]
    delta = mi8["acc_top1"] - m32["acc_top1"]
    linhas_md.append(
        f"| **MobileNetV2 $\\alpha={a:.2f}$ (`{c['img_size'][0]}x{c['img_size'][1]}`)** | "
        f"{m32['acc_top1']:.2f}% | **{mi8['acc_top1']:.2f}%** | `{delta:+.2f}%` | "
        f"{mi8['acc_top2']:.2f}% | {mi8['f1_macro']:.2f}% | "
        f"{mi8['precision_macro']:.2f}% / {mi8['recall_macro']:.2f}% | "
        f"{mi8['kappa']:.2f}% | {mi8['mcc']:.2f}% | "
        f"{mi8['conf_corretos']:.1f}% / {mi8['conf_errados']:.1f}% | {mi8['ece']:.2f}% |"
    )

linhas_md.extend([
    "",
    "---",
    "",
    "## 4. Recall (%) por Patologia / Classe (Float32 vs. Int8 QAT)",
    "",
    "| Classe / Patologia | Val | $\\alpha=0.35$ F32 | $\\alpha=0.35$ Int8 | $\\Delta_{0.35}$ | $\\alpha=0.50$ F32 | $\\alpha=0.50$ Int8 | $\\Delta_{0.50}$ | $\\alpha=0.75$ F32 | $\\alpha=0.75$ Int8 | $\\Delta_{0.75}$ |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |",
])

for i, cls in enumerate(class_names):
    linhas_md.append(
        f"| **{cls}** | {contagem_val_limpo[cls]} | "
        f"{metricas_f32[0.35]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.35]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.35]['acc_por_classe'][i] - metricas_f32[0.35]['acc_por_classe'][i]:+.1f}% | "
        f"{metricas_f32[0.50]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.50]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.50]['acc_por_classe'][i] - metricas_f32[0.50]['acc_por_classe'][i]:+.1f}% | "
        f"{metricas_f32[0.75]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.75]['acc_por_classe'][i]:.1f}% | {metricas_int8[0.75]['acc_por_classe'][i] - metricas_f32[0.75]['acc_por_classe'][i]:+.1f}% |"
    )

linhas_md.extend([
    "",
    "---",
    "",
    "## 5. Matriz de Co-Design Hardware/Software no ESP32-S3 N8R16",
    "",
    "| Modelo (Int8 QAT) | Resolução | Params | MMACs | Flash `.tflite` | Arena PSRAM Est. | Scratch SRAM `esp-nn` | Latência ESP32-S3 Est. | Energia | NetScore ($\\Omega$) | Ops C++ OK? |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |",
])

for a in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[a]
    linhas_md.append(
        f"| `MobileNetV2 α={a:.2f}` | `{hw['img_size'][0]}x{hw['img_size'][1]}` | "
        f"{hw['total_params']/1e3:.1f}k | {hw['mmacs']:.1f}M | "
        f"{hw['kb_size']:.1f} KB ({hw['flash_usage_pct']:.1f}%) | "
        f"{hw['estimated_arena_kb']:.0f} KB | "
        f"{hw['esp_nn_scratch_kb']:.1f} KB ({'OK' if hw['fits_sram_48kb'] else 'EXCEDE'}) | "
        f"{hw['esp32s3_latency_pure_ms']:.0f} ms ({hw['esp32s3_fps']:.2f} FPS) | "
        f"{hw['energy_mj']:.0f} mJ | {hw['netscore_db']:.1f} dB | "
        f"{'Sim' if (hw['firmware_ops_ok'] and hw['is_pure_int8']) else 'Não'} |"
    )

linhas_md.extend([
    "",
    "---",
    "",
    "## 6. Decisão Pareto e Metadados do Header C/C++ Exportado",
    "",
    f"- **Modelo Selecionado:** `MobileNetV2 α={alpha_escolhido:.2f}` (`{MODEL_ESCOLHIDO}`)",
    f"- **Header Gerado:** `{NOME_SAIDA}.h` (`{len(dados_tflite)/1024:.2f} KB` | `MODEL_INPUT_WIDTH={cfg_escolhido['img_size'][1]}`, `MODEL_INPUT_HEIGHT={cfg_escolhido['img_size'][0]}`)",
    f"- **Arena PSRAM Recomendada:** `{ARENA_RECOMENDADA_KB} KB`",
    f"- **Input Quantization:** `scale={in_scale:.8f}`, `zero_point={in_zp}`",
    f"- **Output Quantization:** `scale={out_scale:.8f}`, `zero_point={out_zp}`",
    "",
])

conteudo_relatorio_v10 = "\n".join(linhas_md)
with open(REPORT_MD_PATH, "w", encoding="utf-8") as f_md:
    f_md.write(conteudo_relatorio_v10)

print(f"✅ Relatório salvo com sucesso em: {os.path.abspath(REPORT_MD_PATH)}")
for pasta_drive in ["/content/drive/MyDrive/tcc/reports", "/content/drive/MyDrive"]:
    if os.path.isdir(os.path.dirname(pasta_drive)) or os.path.isdir(pasta_drive):
        try:
            os.makedirs(pasta_drive, exist_ok=True)
            destino_drive = os.path.join(pasta_drive, REPORT_MD_PATH)
            shutil.copy2(REPORT_MD_PATH, destino_drive)
            print(f"☁️ Cópia salva no Google Drive em: {destino_drive}")
            break
        except Exception as e:
            print(f"ℹ️ Não foi possível copiar para o Drive ({pasta_drive}): {e}")

In [ ]:
# Download dos modelos .tflite gerados (α=0.35, 0.50, 0.75), header .h e training_reports_v10.md
try:
    from google.colab import files
    for alpha in [0.35, 0.50, 0.75]:
        files.download(auditoria_hw[alpha]["tflite_path"])
    files.download(f"{NOME_SAIDA}.h")
    files.download(REPORT_MD_PATH)
except ImportError:
    print("ℹ️ Ambiente fora do Google Colab; arquivos salvos localmente no diretório de trabalho.")